# Pretraining notebook

Original model, batching, optimizer, checkpoint, smoke-test and training cells retained in their original order. Standalone input/split setup is included. Run top to bottom in the same project environment. Dataset and BindingDB audits are in the separate diagnostics notebook.

# **PRETRAINING WITH NO TOXICITY OBJECTIVE**

## Environment and input-file verification

In [13]:
from pathlib import Path
import os
import sys
import importlib
from itertools import islice

In [14]:
PROJECT_ROOT = Path(
    os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
).expanduser().resolve()

PREPROCESSING_ROOT = PROJECT_ROOT / "REWORKED_CODE" / "PREPROCESSING_OUTPUT"

CHEMBL_DIR = PREPROCESSING_ROOT / "chembl"
CLEAN_TXT = CHEMBL_DIR / "chembl_clean.txt"
CLEAN_CSV = CHEMBL_DIR / "chembl_clean.csv"

ATOM_TYPES_PATH = (
    PREPROCESSING_ROOT / "vocabulary" / "atom_types.txt"
)

# Do not choose the dataset until its format is verified.
CHEMBL_PATH = None

## ChEMBL Dataset and Vocabulary Inspection

Objective: Understand the actual structure of the cleaned ChEMBL datasets and the final atom vocabulary before implementing the data loader.

In [21]:
from collections import Counter
import hashlib

print("=" * 65)
print("2.3 — FINAL ATOM VOCABULARY")
print("=" * 65)

with ATOM_TYPES_PATH.open("r", encoding="utf-8-sig") as f:
    raw_rows = [
        line.strip()
        for line in f
        if line.strip()
    ]

atom_types = []

for row_num, row in enumerate(raw_rows, 1):
    fields = [x.strip() for x in row.split(",")]

    if len(fields) != 3:
        raise ValueError(
            f"Invalid vocabulary entry at row {row_num}: {row!r}"
        )

    symbol = fields[0]
    formal_charge = int(fields[1])
    explicit_h = int(fields[2])

    atom_types.append((symbol, formal_charge, explicit_h))

duplicates = [
    atom for atom, count in Counter(atom_types).items()
    if count > 1
]

ATOM_VOCAB_HASH = hashlib.sha256(
    ATOM_TYPES_PATH.read_bytes()
).hexdigest()

print(f"Vocabulary path : {ATOM_TYPES_PATH}")
print(f"Total entries   : {len(atom_types)}")
print(f"Unique entries  : {len(set(atom_types))}")
print(f"Duplicates      : {len(duplicates)}")
print(f"SHA256          : {ATOM_VOCAB_HASH}")

print("\nAtom types (first 15):")
for i, atom in enumerate(atom_types[:15]):
    print(f"  {i:3d}: {atom}")

print("\nAtom types (last 5):")
for i in range(max(0, len(atom_types) - 5), len(atom_types)):
    print(f"  {i:3d}: {atom_types[i]}")

print("\nVocabulary validation:")
assert atom_types, "Empty atom vocabulary"
assert not duplicates, f"Duplicate atom types: {duplicates}"

print("STATUS: PASS")

2.3 — FINAL ATOM VOCABULARY
Vocabulary path : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/vocabulary/atom_types.txt
Total entries   : 55
Unique entries  : 55
Duplicates      : 0
SHA256          : 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

Atom types (first 15):
    0: ('B', -1, 0)
    1: ('B', -1, 3)
    2: ('B', 0, 0)
    3: ('Br', 0, 0)
    4: ('C', -1, 0)
    5: ('C', -1, 1)
    6: ('C', 0, 0)
    7: ('C', 0, 1)
    8: ('C', 0, 2)
    9: ('C', 0, 3)
   10: ('C', 1, 0)
   11: ('Cl', 0, 0)
   12: ('Cl', 1, 0)
   13: ('F', 0, 0)
   14: ('Fe', -2, 0)

Atom types (last 5):
   50: ('Se', 0, 0)
   51: ('Si', 0, 0)
   52: ('Sn', 0, 0)
   53: ('Te', 0, 0)
   54: ('V', 0, 0)

Vocabulary validation:
STATUS: PASS


## Reproducible Train/Validation Split

In [33]:
import hashlib
import numpy as np

print("=" * 65)
print("6.1 — PRETRAINING SPLIT CONFIGURATION")
print("=" * 65)

# Dataset configuration
PRETRAIN_DATA_PATH = CLEAN_TXT
PRETRAIN_VOCAB_PATH = ATOM_TYPES_PATH

EXPECTED_MOLECULES = 1_090_506
EXPECTED_DATA_SHA256 = (
    "a26351a9e8438e633c3824a21f3874609e71dc7d3a05c2f2055f7081c85d4b5b"
)
EXPECTED_VOCAB_SHA256 = (
    "3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f"
)

# Split configuration
SPLIT_SEED = 42
VALIDATION_FRACTION = 0.05
TRAIN_FRACTION = 1.0 - VALIDATION_FRACTION

# Training objective
OBJECTIVE = "unconditional_nll"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

data_hash = sha256_file(PRETRAIN_DATA_PATH)
vocab_hash = sha256_file(PRETRAIN_VOCAB_PATH)

assert data_hash == EXPECTED_DATA_SHA256, (
    "ChEMBL dataset changed since the full audit."
)
assert vocab_hash == EXPECTED_VOCAB_SHA256, (
    "Atom vocabulary changed since the full audit."
)
assert len(atom_types) == 55, "Unexpected vocabulary size."

print("\nINPUT VERIFICATION")
print(f"Dataset          : {PRETRAIN_DATA_PATH.name}")
print(f"Expected records : {EXPECTED_MOLECULES:,}")
print(f"Atom types       : {len(atom_types)}")
print(f"Dataset SHA256   : {data_hash}")
print(f"Vocabulary SHA256: {vocab_hash}")

print("\nSPLIT CONFIGURATION")
print(f"Training fraction  : {TRAIN_FRACTION:.1%}")
print(f"Validation fraction: {VALIDATION_FRACTION:.1%}")
print(f"Random seed        : {SPLIT_SEED}")
print(f"Objective          : {OBJECTIVE}")

print("\nSTATUS: PASS — configuration and fingerprints verified.")

6.1 — PRETRAINING SPLIT CONFIGURATION

INPUT VERIFICATION
Dataset          : chembl_clean.txt
Expected records : 1,090,506
Atom types       : 55
Dataset SHA256   : a26351a9e8438e633c3824a21f3874609e71dc7d3a05c2f2055f7081c85d4b5b
Vocabulary SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

SPLIT CONFIGURATION
Training fraction  : 95.0%
Validation fraction: 5.0%
Random seed        : 42
Objective          : unconditional_nll

STATUS: PASS — configuration and fingerprints verified.


In [34]:
print("=" * 65)
print("6.2 — TRAIN / VALIDATION SPLIT")
print("=" * 65)

N_MOLECULES = EXPECTED_MOLECULES

rng = np.random.default_rng(SPLIT_SEED)

n_val = round(N_MOLECULES * VALIDATION_FRACTION)

# Select held-out molecule indices.
val_indices = np.sort(
    rng.choice(
        N_MOLECULES,
        size=n_val,
        replace=False
    )
).astype(np.int32)

# Remaining molecules become training data.
train_mask = np.ones(N_MOLECULES, dtype=bool)
train_mask[val_indices] = False

train_indices = np.flatnonzero(train_mask).astype(np.int32)

# Verify partition properties.
assert len(train_indices) + len(val_indices) == N_MOLECULES
assert len(np.unique(val_indices)) == len(val_indices)
assert np.all(~train_mask[val_indices])
assert np.all(train_mask[train_indices])
assert len(train_indices) == int(train_mask.sum())

# Fingerprint the selected validation indices.
SPLIT_HASH = hashlib.sha256(
    val_indices.tobytes()
).hexdigest()

print("\nSPLIT RESULTS")
print(f"Total molecules  : {N_MOLECULES:,}")
print(f"Training set     : {len(train_indices):,}")
print(f"Validation set   : {len(val_indices):,}")

print(f"\nTraining ratio   : {len(train_indices)/N_MOLECULES:.4%}")
print(f"Validation ratio : {len(val_indices)/N_MOLECULES:.4%}")

print("\nINTEGRITY CHECKS")
print(f"All records assigned       : PASS")
print(f"No shared row indices      : PASS")
print(f"No duplicate val indices   : PASS")
print(f"Validation index SHA256    : {SPLIT_HASH}")

print("\nFIRST 10 VALIDATION INDICES")
print(val_indices[:10].tolist())

print("\nSTATUS: PASS — deterministic split created.")

6.2 — TRAIN / VALIDATION SPLIT

SPLIT RESULTS
Total molecules  : 1,090,506
Training set     : 1,035,981
Validation set   : 54,525

Training ratio   : 95.0000%
Validation ratio : 5.0000%

INTEGRITY CHECKS
All records assigned       : PASS
No shared row indices      : PASS
No duplicate val indices   : PASS
Validation index SHA256    : c0126bcc5ba1bc930ef6a947ee87f73c8cb61c01940a65eb60b96179ab517526

FIRST 10 VALIDATION INDICES
[0, 3, 14, 20, 45, 48, 53, 114, 133, 144]

STATUS: PASS — deterministic split created.


In [36]:
import json
import os
import tempfile

print("=" * 65)
print("6.4 — SAVE REPRODUCIBLE DATASET SPLIT")
print("=" * 65)

SPLIT_DIR = (
    PROJECT_ROOT
    / "REWORKED_CODE"
    / "PRETRAINING_OUTPUT"
    / "no_toxicity"
    / "splits"
)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_name = (
    f"chembl_clean_seed{SPLIT_SEED}"
    f"_val5_{SPLIT_HASH[:12]}"
)

SPLIT_NPZ_PATH = SPLIT_DIR / f"{split_name}.npz"
SPLIT_MANIFEST_PATH = SPLIT_DIR / f"{split_name}.json"

manifest = {
    "schema_version": 1,
    "objective": "cdgcn_nll",
    "dataset_path": str(PRETRAIN_DATA_PATH.resolve()),
    "dataset_sha256": data_hash,
    "vocabulary_path": str(PRETRAIN_VOCAB_PATH.resolve()),
    "vocabulary_sha256": vocab_hash,
    "total_molecules": N_MOLECULES,
    "train_molecules": len(train_indices),
    "validation_molecules": len(val_indices),
    "split_seed": SPLIT_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "split_method": "random_molecule_holdout",
    "validation_indices_sha256": SPLIT_HASH,
}

assert len(train_indices) + len(val_indices) == N_MOLECULES

if SPLIT_NPZ_PATH.exists() or SPLIT_MANIFEST_PATH.exists():
    assert SPLIT_NPZ_PATH.is_file()
    assert SPLIT_MANIFEST_PATH.is_file()

    with SPLIT_MANIFEST_PATH.open("r") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == manifest, "Existing manifest differs."

    with np.load(SPLIT_NPZ_PATH, allow_pickle=False) as existing:
        assert np.array_equal(existing["train_indices"], train_indices)
        assert np.array_equal(existing["val_indices"], val_indices)

    print("Existing split verified; no files overwritten.")

else:
    # Write temporary files first, then publish the completed artifacts.
    temp_npz = None
    temp_json = None

    try:
        with tempfile.NamedTemporaryFile(
            dir=SPLIT_DIR, suffix=".npz", delete=False
        ) as f:
            temp_npz = Path(f.name)
            np.savez_compressed(
                f,
                train_indices=train_indices,
                val_indices=val_indices,
            )

        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8",
            dir=SPLIT_DIR, suffix=".json", delete=False
        ) as f:
            temp_json = Path(f.name)
            json.dump(manifest, f, indent=2, sort_keys=True)

        os.replace(temp_npz, SPLIT_NPZ_PATH)
        os.replace(temp_json, SPLIT_MANIFEST_PATH)

        print("New split artifacts created.")

    finally:
        for temporary in [temp_npz, temp_json]:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

print("\nSAVED ARTIFACTS")
print(f"Indices  : {SPLIT_NPZ_PATH}")
print(f"Manifest : {SPLIT_MANIFEST_PATH}")
print(f"Train    : {len(train_indices):,}")
print(f"Val      : {len(val_indices):,}")
print(f"Hash     : {SPLIT_HASH}")

print("\nSTATUS: Split persistence completed.")

6.4 — SAVE REPRODUCIBLE DATASET SPLIT
New split artifacts created.

SAVED ARTIFACTS
Indices  : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/splits/chembl_clean_seed42_val5_c0126bcc5ba1.npz
Manifest : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/splits/chembl_clean_seed42_val5_c0126bcc5ba1.json
Train    : 1,035,981
Val      : 54,525
Hash     : c0126bcc5ba1bc930ef6a947ee87f73c8cb61c01940a65eb60b96179ab517526

STATUS: Split persistence completed.


## Molecular Graph Representation and Generation Paths.

In [38]:
from pathlib import Path
from rdkit import Chem

print("=" * 65)
print("7.1 — ATOM AND BOND VOCABULARY MAPPING")
print("=" * 65)

MAX_ITER_PER_MOL = 120
RECEPTIVE_FIELD = 2

class MoleculeSpec:
    def __init__(self, vocabulary_path):
        self.vocabulary_path = Path(vocabulary_path)

        with self.vocabulary_path.open(
            "r", encoding="utf-8-sig"
        ) as f:
            self.atom_types = [
                (parts[0], int(parts[1]), int(parts[2]))
                for line in f
                if (parts := [x.strip() for x in line.strip().split(",")])
                and len(parts) == 3
            ]

        assert len(self.atom_types) == 55
        assert len(self.atom_types) == len(set(self.atom_types))

        self.atom_index = {
            atom: i for i, atom in enumerate(self.atom_types)
        }

        # Preserve the original CDGCN bond ordering.
        self.bond_orders = [
            Chem.BondType.AROMATIC,
            Chem.BondType.SINGLE,
            Chem.BondType.DOUBLE,
            Chem.BondType.TRIPLE,
        ]

        self.bond_index = {
            bond: i for i, bond in enumerate(self.bond_orders)
        }

        self.num_atom_types = len(self.atom_types)
        self.num_bond_types = len(self.bond_orders)
        self.max_iter = MAX_ITER_PER_MOL

    def get_atom_type(self, atom):
        key = (
            atom.GetSymbol(),
            atom.GetFormalCharge(),
            atom.GetNumExplicitHs(),
        )
        if key not in self.atom_index:
            raise ValueError(f"Unknown atom type: {key}")
        return self.atom_index[key]

    def get_bond_type(self, bond):
        bond_type = bond.GetBondType()
        if bond_type not in self.bond_index:
            raise ValueError(f"Unknown bond type: {bond_type}")
        return self.bond_index[bond_type]

    def index_to_atom(self, index):
        symbol, charge, explicit_h = self.atom_types[int(index)]

        atom = Chem.Atom(symbol)
        atom.SetFormalCharge(charge)
        atom.SetNumExplicitHs(explicit_h)
        return atom

    def index_to_bond(self, mol, begin, end, index):
        mol.AddBond(
            int(begin),
            int(end),
            self.bond_orders[int(index)]
        )

MOL_SPEC = MoleculeSpec(PRETRAIN_VOCAB_PATH)

7.1 — ATOM AND BOND VOCABULARY MAPPING


In [39]:
def get_mol_spec():
    return MOL_SPEC

assert MOL_SPEC.atom_types == atom_types

print("\nVOCABULARY SUMMARY")
print(f"Atom types         : {MOL_SPEC.num_atom_types}")
print(f"Bond types         : {MOL_SPEC.num_bond_types}")
print(f"Maximum steps      : {MOL_SPEC.max_iter}")
print(f"Receptive field    : {RECEPTIVE_FIELD}")

print("\nBOND TYPE MAPPING")
for bond, index in MOL_SPEC.bond_index.items():
    print(f"  {index}: {bond}")

print("\nATOM MAPPING EXAMPLES")
for i in range(min(10, MOL_SPEC.num_atom_types)):
    print(f"  {i}: {MOL_SPEC.atom_types[i]}")

print("\nROUND-TRIP VOCABULARY CHECK")
for index in range(MOL_SPEC.num_atom_types):
    atom = MOL_SPEC.index_to_atom(index)
    assert MOL_SPEC.get_atom_type(atom) == index

print(f"Verified {MOL_SPEC.num_atom_types}/55 atom mappings.")
print("\nSTATUS: PASS — vocabulary mapping established.")


VOCABULARY SUMMARY
Atom types         : 55
Bond types         : 4
Maximum steps      : 120
Receptive field    : 2

BOND TYPE MAPPING
  0: AROMATIC
  1: SINGLE
  2: DOUBLE
  3: TRIPLE

ATOM MAPPING EXAMPLES
  0: ('B', -1, 0)
  1: ('B', -1, 3)
  2: ('B', 0, 0)
  3: ('Br', 0, 0)
  4: ('C', -1, 0)
  5: ('C', -1, 1)
  6: ('C', 0, 0)
  7: ('C', 0, 1)
  8: ('C', 0, 2)
  9: ('C', 0, 3)

ROUND-TRIP VOCABULARY CHECK
Verified 55/55 atom mappings.

STATUS: PASS — vocabulary mapping established.


In [40]:
import networkx as nx
from itertools import islice

print("=" * 65)
print("7.2 — SMILES TO GRAPH CONVERSION")
print("=" * 65)

def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    atom_types_encoded = [
        MOL_SPEC.get_atom_type(atom)
        for atom in mol.GetAtoms()
    ]

    atom_ranks = list(Chem.CanonicalRankAtoms(mol))

    bonds = []
    bond_types_encoded = []

    for bond in mol.GetBonds():
        bonds.append([
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
        ])
        bond_types_encoded.append(
            MOL_SPEC.get_bond_type(bond)
        )

    graph = nx.Graph()
    graph.add_nodes_from(range(mol.GetNumAtoms()))
    graph.add_edges_from(bonds)

    if not nx.is_connected(graph):
        raise ValueError("Disconnected molecular graph")

    if mol.GetNumBonds() + 1 > MAX_ITER_PER_MOL:
        raise ValueError("Molecule exceeds generation step limit")

    assert graph.number_of_nodes() == len(atom_types_encoded)
    assert graph.number_of_edges() == len(bond_types_encoded)
    assert len(atom_ranks) == len(atom_types_encoded)

    return (
        graph,
        atom_types_encoded,
        atom_ranks,
        bonds,
        bond_types_encoded,
    )

# Test known structures and real ChEMBL molecules.
with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    real_examples = [
        line.strip() for line in islice(f, 5)
    ]

TEST_SMILES = [
    "CCO",
    "c1ccccc1",
    "C1CCCCC1",
    "CC#N",
] + real_examples

for i, smiles in enumerate(TEST_SMILES, 1):
    graph, encoded_atoms, ranks, bonds, encoded_bonds = (
        get_graph_from_smiles(smiles)
    )

    print(f"\nMOLECULE {i}")
    print("-" * 45)
    print(f"SMILES       : {smiles[:100]}")
    print(f"Atoms        : {graph.number_of_nodes()}")
    print(f"Bonds        : {graph.number_of_edges()}")
    print(f"Connected    : {nx.is_connected(graph)}")
    print(f"Atom IDs     : {encoded_atoms[:15]}")
    print(f"Atom ranks   : {ranks[:15]}")
    print(f"Bond IDs     : {encoded_bonds[:15]}")
    print(f"Steps needed : {len(bonds) + 1}")

print("\nSTATUS: PASS — test molecular graphs constructed.")

7.2 — SMILES TO GRAPH CONVERSION

MOLECULE 1
---------------------------------------------
SMILES       : CCO
Atoms        : 3
Bonds        : 2
Connected    : True
Atom IDs     : [6, 6, 32]
Atom ranks   : [0, 2, 1]
Bond IDs     : [1, 1]
Steps needed : 3

MOLECULE 2
---------------------------------------------
SMILES       : c1ccccc1
Atoms        : 6
Bonds        : 6
Connected    : True
Atom IDs     : [6, 6, 6, 6, 6, 6]
Atom ranks   : [3, 1, 0, 2, 4, 5]
Bond IDs     : [0, 0, 0, 0, 0, 0]
Steps needed : 7

MOLECULE 3
---------------------------------------------
SMILES       : C1CCCCC1
Atoms        : 6
Bonds        : 6
Connected    : True
Atom IDs     : [6, 6, 6, 6, 6, 6]
Atom ranks   : [3, 1, 0, 2, 4, 5]
Bond IDs     : [1, 1, 1, 1, 1, 1]
Steps needed : 7

MOLECULE 4
---------------------------------------------
SMILES       : CC#N
Atoms        : 3
Bonds        : 2
Connected    : True
Atom IDs     : [6, 6, 24]
Atom ranks   : [0, 2, 1]
Bond IDs     : [1, 3]
Steps needed : 3

MOLECULE 5
--

In [41]:
from collections import Counter

print("=" * 65)
print("7.3 — MOLECULAR GRAPH RECONSTRUCTION TEST")
print("=" * 65)

def reconstruct_molecule(atom_ids, bonds, bond_ids):
    rw_mol = Chem.RWMol()

    for atom_id in atom_ids:
        rw_mol.AddAtom(
            MOL_SPEC.index_to_atom(atom_id)
        )

    for (begin, end), bond_id in zip(bonds, bond_ids):
        MOL_SPEC.index_to_bond(
            rw_mol, begin, end, bond_id
        )

    mol = rw_mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol

results = Counter()
failures = []

# Include the known examples and a larger deterministic
# sample of real dataset molecules.
with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    dataset_examples = [
        line.strip() for line in islice(f, 1000)
    ]

ROUNDTRIP_SMILES = TEST_SMILES + dataset_examples

for smiles in ROUNDTRIP_SMILES:
    results["tested"] += 1

    try:
        original = Chem.MolFromSmiles(smiles)

        _, atom_ids, _, bonds, bond_ids = (
            get_graph_from_smiles(smiles)
        )

        reconstructed = reconstruct_molecule(
            atom_ids, bonds, bond_ids
        )

        original_plain = Chem.MolToSmiles(
            original, canonical=True, isomericSmiles=False
        )

        reconstructed_plain = Chem.MolToSmiles(
            reconstructed, canonical=True, isomericSmiles=False
        )

        original_isomeric = Chem.MolToSmiles(
            original, canonical=True, isomericSmiles=True
        )

        reconstructed_isomeric = Chem.MolToSmiles(
            reconstructed, canonical=True, isomericSmiles=True
        )

        if original_plain == reconstructed_plain:
            results["structure_match"] += 1
        else:
            results["structure_mismatch"] += 1
            if len(failures) < 10:
                failures.append({
                    "original": original_plain,
                    "reconstructed": reconstructed_plain,
                })

        if original_isomeric == reconstructed_isomeric:
            results["isomeric_match"] += 1
        else:
            results["isomeric_mismatch"] += 1

    except Exception as exc:
        results["errors"] += 1
        if len(failures) < 10:
            failures.append({
                "smiles": smiles,
                "error": str(exc),
            })

print("\nROUND-TRIP RESULTS")
for key in [
    "tested",
    "structure_match",
    "structure_mismatch",
    "isomeric_match",
    "isomeric_mismatch",
    "errors",
]:
    print(f"{key:25s}: {results[key]:,}")

print("\nSTRUCTURAL RECONSTRUCTION RATE")
print(
    f"{100 * results['structure_match'] / results['tested']:.2f}%"
)

print("\nFAILURE EXAMPLES")
for example in failures:
    print(example)
if not failures:
    print("None")

print("\nSTATUS")
if results["structure_mismatch"] == 0 and results["errors"] == 0:
    print("PASS — tested graphs reconstruct without structural mismatches.")
else:
    print("REVIEW REQUIRED — graph reconstruction differences detected.")

if results["isomeric_mismatch"] > 0:
    print(
        "NOTE — isomeric differences may reflect stereochemistry "
        "not represented by the CDGCN graph encoding."
    )

7.3 — MOLECULAR GRAPH RECONSTRUCTION TEST

ROUND-TRIP RESULTS
tested                   : 1,009
structure_match          : 1,009
structure_mismatch       : 0
isomeric_match           : 731
isomeric_mismatch        : 278
errors                   : 0

STRUCTURAL RECONSTRUCTION RATE
100.00%

FAILURE EXAMPLES
None

STATUS
PASS — tested graphs reconstruct without structural mismatches.
NOTE — isomeric differences may reflect stereochemistry not represented by the CDGCN graph encoding.


## Generation Path Construction

In [42]:
import math
import random
import networkx as nx

print("=" * 65)
print("8.1 — GENERATION PATH TRAVERSAL")
print("=" * 65)

K_PATHS = 5
P_ALPHA = 0.8

def traverse_graph(
    graph,
    atom_ranks,
    *,
    p=P_ALPHA,
    rng=None,
    return_trace=False
):
    """
    CDGCN-style stochastic depth-first graph traversal.

    Returns:
        step_ids: original atom index -> generation order
        log_q: log probability of sampled path under Q_alpha

    With return_trace=True, also returns decision records.
    """

    if not (0.0 < p < 1.0):
        raise ValueError("p must be strictly between 0 and 1")

    n = graph.number_of_nodes()

    if n == 0 or set(graph.nodes) != set(range(n)):
        raise ValueError("Expected contiguous atom IDs 0..n-1")

    if len(atom_ranks) != n:
        raise ValueError("Atom rank count does not match graph")

    if not nx.is_connected(graph):
        raise ValueError("Disconnected molecular graph")

    rng = rng if rng is not None else random

    step_ids = [-1] * n
    trace = []
    log_q = 0.0
    next_step = 0

    def visit(current_node):
        nonlocal log_q, next_step

        while True:
            if current_node is None:
                available = range(n)
            else:
                available = graph.neighbors(current_node)

            choices = sorted(
                (v for v in available if step_ids[v] == -1),
                key=lambda v: (atom_ranks[v], v)
            )

            if not choices:
                return

            if len(choices) == 1:
                selected = choices[0]
                probability = 1.0
                mode = "forced"

            elif rng.random() < p:
                selected = choices[0]
                probability = p
                mode = "canonical"

            else:
                selected = choices[
                    1 + rng.randrange(len(choices) - 1)
                ]
                probability = (1.0 - p) / (len(choices) - 1)
                mode = "random-other"

            step_ids[selected] = next_step
            next_step += 1

            log_q += math.log(probability)

            trace.append({
                "step": step_ids[selected],
                "parent": current_node,
                "node": selected,
                "choices": len(choices),
                "mode": mode,
                "probability": probability,
            })

            visit(selected)

    visit(None)

    assert sorted(step_ids) == list(range(n))
    assert len(trace) == n
    assert math.isfinite(log_q)

    if return_trace:
        return step_ids, float(log_q), trace

    return step_ids, float(log_q)


print(f"K paths per molecule : {K_PATHS}")
print(f"Alpha                : {P_ALPHA}")
print("Traversal            : Stochastic DFS")
print("Canonical ordering   : RDKit atom ranks")
print("Randomness           : Injectable RNG")
print("Probability output   : log Q_alpha(T | G)")

print("\nSTATUS: Traversal function defined.")

8.1 — GENERATION PATH TRAVERSAL
K paths per molecule : 5
Alpha                : 0.8
Traversal            : Stochastic DFS
Canonical ordering   : RDKit atom ranks
Randomness           : Injectable RNG
Probability output   : log Q_alpha(T | G)

STATUS: Traversal function defined.


In [43]:
from collections import Counter

print("=" * 65)
print("8.2 — TRAVERSAL CORRECTNESS TEST")
print("=" * 65)

TRAVERSAL_TEST_SMILES = [
    "CCO",
    "CC(C)O",
    "c1ccccc1",
    "C1CCCCC1",
    "CC(C)C(C)C",
] + real_examples[:2]

test_results = Counter()

for i, smiles in enumerate(TRAVERSAL_TEST_SMILES, 1):
    graph, atom_ids, atom_ranks, bonds, bond_ids = (
        get_graph_from_smiles(smiles)
    )

    seed = 42 + i

    steps, log_q, trace = traverse_graph(
        graph,
        atom_ranks,
        p=P_ALPHA,
        rng=random.Random(seed),
        return_trace=True
    )

    # Repeat with identical seed.
    steps_again, log_q_again, trace_again = traverse_graph(
        graph,
        atom_ranks,
        p=P_ALPHA,
        rng=random.Random(seed),
        return_trace=True
    )

    n_atoms = graph.number_of_nodes()
    n_bonds = graph.number_of_edges()

    assert steps == steps_again
    assert log_q == log_q_again
    assert trace == trace_again

    assert len(steps) == n_atoms
    assert sorted(steps) == list(range(n_atoms))

    # Check every selected atom has a valid DFS parent.
    for entry in trace:
        if entry["parent"] is None:
            assert entry["step"] == 0
        else:
            assert graph.has_edge(
                entry["parent"], entry["node"]
            )
            assert (
                steps[entry["parent"]] < steps[entry["node"]]
            )

    # Independently sum the recorded decision probabilities.
    calculated_log_q = math.fsum(
        math.log(entry["probability"])
        for entry in trace
    )

    assert math.isclose(
        calculated_log_q,
        log_q,
        rel_tol=1e-10,
        abs_tol=1e-10
    )

    append_count = n_atoms - 1
    connect_count = n_bonds - append_count
    post_init_actions = n_bonds + 1

    assert connect_count >= 0
    assert post_init_actions <= MAX_ITER_PER_MOL

    test_results["passed"] += 1

    print(f"\nMOLECULE {i}")
    print("-" * 48)
    print(f"SMILES              : {smiles[:90]}")
    print(f"Atoms               : {n_atoms}")
    print(f"Bonds               : {n_bonds}")
    print(f"Generation order    : {steps}")
    print(f"Initial atom ID     : {steps.index(0)}")
    print(f"Append actions      : {append_count}")
    print(f"Connect actions     : {connect_count}")
    print(f"End actions         : 1")
    print(f"Post-init actions   : {post_init_actions}")
    print(f"log Q_alpha         : {log_q:.6f}")
    print(f"Stochastic decisions: {sum(t['choices'] > 1 for t in trace)}")
    print("Reproducibility     : PASS")

    if i <= 2:
        print("\nFirst traversal decisions:")
        for entry in trace[:8]:
            print(
                f"  step={entry['step']:2d} "
                f"atom={entry['node']:2d} "
                f"parent={str(entry['parent']):>4s} "
                f"choices={entry['choices']:2d} "
                f"mode={entry['mode']:12s} "
                f"prob={entry['probability']:.4f}"
            )

print("\n" + "=" * 65)
print("TRAVERSAL TEST SUMMARY")
print("=" * 65)
print(f"Molecules tested : {len(TRAVERSAL_TEST_SMILES)}")
print(f"Tests passed     : {test_results['passed']}")
print("Atom visitation  : PASS")
print("Graph connectivity: PASS")
print("Log probability  : PASS")
print("Reproducibility  : PASS")
print("\nSTATUS: PASS — traversal invariants verified.")

8.2 — TRAVERSAL CORRECTNESS TEST

MOLECULE 1
------------------------------------------------
SMILES              : CCO
Atoms               : 3
Bonds               : 2
Generation order    : [0, 1, 2]
Initial atom ID     : 0
Append actions      : 2
Connect actions     : 0
End actions         : 1
Post-init actions   : 3
log Q_alpha         : -0.223144
Stochastic decisions: 1
Reproducibility     : PASS

First traversal decisions:
  step= 0 atom= 0 parent=None choices= 3 mode=canonical    prob=0.8000
  step= 1 atom= 1 parent=   0 choices= 1 mode=forced       prob=1.0000
  step= 2 atom= 2 parent=   1 choices= 1 mode=forced       prob=1.0000

MOLECULE 2
------------------------------------------------
SMILES              : CC(C)O
Atoms               : 4
Bonds               : 3
Generation order    : [0, 1, 2, 3]
Initial atom ID     : 0
Append actions      : 3
Connect actions     : 0
End actions         : 1
Post-init actions   : 4
log Q_alpha         : -0.446287
Stochastic decisions: 2
Reprodu

In [44]:
print("=" * 65)
print("8.3 — PROPOSAL DISTRIBUTION TEST")
print("=" * 65)

TEST_MOLECULE = "CCO"
N_TRIALS = 1000

graph, _, ranks, _, _ = get_graph_from_smiles(
    TEST_MOLECULE
)

canonical_root = min(
    range(len(ranks)),
    key=lambda v: (ranks[v], v)
)

rng = random.Random(2026)

root_counts = Counter()
path_counts = Counter()
log_q_values = []

for _ in range(N_TRIALS):
    steps, log_q, trace = traverse_graph(
        graph,
        ranks,
        p=P_ALPHA,
        rng=rng,
        return_trace=True
    )

    root_counts[trace[0]["node"]] += 1
    path_counts[tuple(steps)] += 1
    log_q_values.append(log_q)

canonical_frequency = (
    root_counts[canonical_root] / N_TRIALS
)

print("\nTEST CONFIGURATION")
print(f"Molecule         : {TEST_MOLECULE}")
print(f"Trials           : {N_TRIALS}")
print(f"Alpha            : {P_ALPHA}")
print(f"Canonical root ID: {canonical_root}")

print("\nROOT SAMPLING DISTRIBUTION")
for node in sorted(root_counts):
    observed = root_counts[node] / N_TRIALS

    expected = (
        P_ALPHA
        if node == canonical_root
        else (1.0 - P_ALPHA) / (graph.number_of_nodes() - 1)
    )

    print(
        f"Atom {node}: "
        f"observed={observed:.3f}, "
        f"expected={expected:.3f}, "
        f"count={root_counts[node]}"
    )

print("\nPATH STATISTICS")
print(f"Unique sampled paths : {len(path_counts)}")
print(f"Minimum log Q        : {min(log_q_values):.6f}")
print(f"Maximum log Q        : {max(log_q_values):.6f}")
print(f"Mean log Q           : {sum(log_q_values)/N_TRIALS:.6f}")

print("\nVALIDATION")
deviation = abs(canonical_frequency - P_ALPHA)

print(f"Canonical root frequency : {canonical_frequency:.4f}")
print(f"Absolute deviation       : {deviation:.4f}")

assert deviation < 0.08, (
    "Canonical root sampling differs substantially from alpha"
)

assert len(path_counts) > 1, (
    "Traversal appears deterministic"
)

assert all(math.isfinite(v) for v in log_q_values)

print("\nSTATUS: PASS — empirical proposal behavior is consistent.")

8.3 — PROPOSAL DISTRIBUTION TEST

TEST CONFIGURATION
Molecule         : CCO
Trials           : 1000
Alpha            : 0.8
Canonical root ID: 0

ROOT SAMPLING DISTRIBUTION
Atom 0: observed=0.794, expected=0.800, count=794
Atom 1: observed=0.113, expected=0.100, count=113
Atom 2: observed=0.093, expected=0.100, count=93

PATH STATISTICS
Unique sampled paths : 4
Minimum log Q        : -3.912023
Maximum log Q        : -0.223144
Mean log Q           : -0.715540

VALIDATION
Canonical root frequency : 0.7940
Absolute deviation       : 0.0060

STATUS: PASS — empirical proposal behavior is consistent.


## Action Sequence Construction and Replay

In [45]:
import numpy as np

print("=" * 65)
print("9.1 — MOLECULAR GRAPH REORDERING")
print("=" * 65)

def single_reorder(X_0, A_0, step_ids):
    """
    Reorder a molecular graph into its generation order.

    X_0: atom-type IDs, shape [num_atoms]
    A_0: [begin, end, bond_type], shape [num_bonds, 3]
    step_ids: original atom ID -> generation-order ID

    Returns:
        X_reordered
        A_reordered

    A bond is oriented as:
        source < target : append
        source > target : connect
    """

    X_0 = np.asarray(X_0, dtype=np.int32)
    A_0 = np.asarray(A_0, dtype=np.int32).reshape(-1, 3)
    step_ids = np.asarray(step_ids, dtype=np.int32)

    n_atoms = len(X_0)

    assert sorted(step_ids.tolist()) == list(range(n_atoms))

    # Reorder atom types by generation order.
    original_indices = np.argsort(step_ids)
    X_reordered = X_0[original_indices]

    # Handle a possible single-atom graph.
    if len(A_0) == 0:
        assert n_atoms == 1
        return X_reordered, A_0.copy()

    # Remap original bond endpoints.
    A_reordered = A_0.copy()

    A_reordered[:, 0] = step_ids[A_0[:, 0]]
    A_reordered[:, 1] = step_ids[A_0[:, 1]]

    max_b = np.max(A_reordered[:, :2], axis=1)
    min_b = np.min(A_reordered[:, :2], axis=1)

    # Original CDGCN ordering.
    order = np.lexsort((-min_b, max_b))
    A_reordered = A_reordered[order]

    # Orient first edge of each new atom as APPEND.
    # Additional edges to that atom are CONNECT.
    oriented_edges = []
    previous_max = -1

    for source, target, bond_type in A_reordered:
        low, high = sorted((int(source), int(target)))

        if high != previous_max:
            oriented_edges.append([low, high, int(bond_type)])
        else:
            oriented_edges.append([high, low, int(bond_type)])

        previous_max = high

    A_reordered = np.asarray(
        oriented_edges, dtype=np.int32
    ).reshape(-1, 3)

    append_targets = [
        int(v) for u, v, _ in A_reordered if u < v
    ]

    assert sorted(append_targets) == list(range(1, n_atoms))
    assert len(A_reordered) == len(A_0)

    return X_reordered, A_reordered


REORDER_TEST_SMILES = [
    "CCO",
    "CC(C)O",
    "c1ccccc1",
    "C1CCCCC1",
    "CC(C)C(C)C",
] + real_examples[:2]

for i, smiles in enumerate(REORDER_TEST_SMILES, 1):
    graph, atom_ids, ranks, bonds, bond_ids = (
        get_graph_from_smiles(smiles)
    )

    steps, log_q = traverse_graph(
        graph,
        ranks,
        p=P_ALPHA,
        rng=random.Random(100 + i)
    )

    X_original = np.asarray(atom_ids, dtype=np.int32)
    A_original = np.column_stack([
        np.asarray(bonds, dtype=np.int32),
        np.asarray(bond_ids, dtype=np.int32),
    ])

    X_reordered, A_reordered = single_reorder(
        X_original, A_original, steps
    )

    append_count = int(np.sum(
        A_reordered[:, 0] < A_reordered[:, 1]
    ))

    connect_count = int(np.sum(
        A_reordered[:, 0] > A_reordered[:, 1]
    ))

    assert append_count == len(atom_ids) - 1
    assert connect_count == len(bonds) - len(atom_ids) + 1

    print(f"\nMOLECULE {i}: {smiles[:75]}")
    print(f"Atoms          : {len(X_reordered)}")
    print(f"Bonds          : {len(A_reordered)}")
    print(f"Append actions : {append_count}")
    print(f"Connect actions: {connect_count}")
    print(f"Initial atom   : {X_reordered[0]}")
    print("First reordered bonds:")

    for edge in A_reordered[:8]:
        action = "APPEND" if edge[0] < edge[1] else "CONNECT"
        print(f"  {edge.tolist()}  {action}")

print("\nSTATUS: PASS — graph reordering verified.")

9.1 — MOLECULAR GRAPH REORDERING

MOLECULE 1: CCO
Atoms          : 3
Bonds          : 2
Append actions : 2
Connect actions: 0
Initial atom   : 6
First reordered bonds:
  [0, 1, 1]  APPEND
  [1, 2, 1]  APPEND

MOLECULE 2: CC(C)O
Atoms          : 4
Bonds          : 3
Append actions : 3
Connect actions: 0
Initial atom   : 6
First reordered bonds:
  [0, 1, 1]  APPEND
  [1, 2, 1]  APPEND
  [1, 3, 1]  APPEND

MOLECULE 3: c1ccccc1
Atoms          : 6
Bonds          : 6
Append actions : 5
Connect actions: 1
Initial atom   : 6
First reordered bonds:
  [0, 1, 0]  APPEND
  [1, 2, 0]  APPEND
  [2, 3, 0]  APPEND
  [3, 4, 0]  APPEND
  [4, 5, 0]  APPEND
  [5, 0, 0]  CONNECT

MOLECULE 4: C1CCCCC1
Atoms          : 6
Bonds          : 6
Append actions : 5
Connect actions: 1
Initial atom   : 6
First reordered bonds:
  [0, 1, 1]  APPEND
  [1, 2, 1]  APPEND
  [2, 3, 1]  APPEND
  [3, 4, 1]  APPEND
  [4, 5, 1]  APPEND
  [5, 0, 1]  CONNECT

MOLECULE 5: CC(C)C(C)C
Atoms          : 6
Bonds          : 5
Append act

In [46]:
print("=" * 65)
print("9.2 — GRAPH STATE AND ACTION EXPANSION")
print("=" * 65)

def single_expand(X_0, A_0):
    """
    Expand a generation-ordered molecule into states.

    Returns:
        action_0       [1]
        X              flattened state atom IDs
        NX             atoms per state
        A              flattened state bonds
        NA             bonds per state
        actions        [num_bonds + 1, 5]
        last_atom_mask flattened positional masks

    Action columns:
        action_type, atom_type, bond_type,
        append_pos, connect_pos
    """

    X_0 = np.asarray(X_0, dtype=np.int32)
    A_0 = np.asarray(A_0, dtype=np.int32).reshape(-1, 3)

    atom_snapshots = []
    edge_snapshots = []
    mask_snapshots = []

    NX = []
    NA = []
    actions = []

    current_atoms = 1
    previous_action_type = 0

    # One pre-action state per bond, plus END state.
    for step in range(len(A_0) + 1):

        NX.append(current_atoms)
        NA.append(step)

        # Graph before the current action.
        atom_snapshots.append(X_0[:current_atoms].copy())
        edge_snapshots.append(A_0[:step].copy())

        mask = np.zeros(current_atoms, dtype=np.int32)
        mask[-1] = (
            1 if previous_action_type == 0 else 2
        )
        mask_snapshots.append(mask)

        if step == len(A_0):
            # END action.
            actions.append([2, 0, 0, 0, 0])
            continue

        source, target, bond_type = map(int, A_0[step])

        if source < target:
            # APPEND a new atom.
            assert target == current_atoms
            assert 0 <= source < current_atoms

            actions.append([
                0,
                int(X_0[target]),
                bond_type,
                source,
                0,
            ])

            current_atoms += 1
            previous_action_type = 0

        else:
            # CONNECT the last appended atom.
            assert source == current_atoms - 1
            assert 0 <= target < source

            actions.append([
                1,
                0,
                bond_type,
                0,
                target,
            ])

            previous_action_type = 1

    assert current_atoms == len(X_0)

    action_0 = np.asarray([X_0[0]], dtype=np.int32)

    X = np.concatenate(atom_snapshots)
    A = np.concatenate(edge_snapshots).reshape(-1, 3)
    last_atom_mask = np.concatenate(mask_snapshots)

    NX = np.asarray(NX, dtype=np.int32)
    NA = np.asarray(NA, dtype=np.int32)
    actions = np.asarray(actions, dtype=np.int32)

    assert len(NX) == len(NA) == len(actions)
    assert len(X) == len(last_atom_mask)
    assert sum(NX) == len(X)
    assert sum(NA) == len(A)

    return (
        action_0, X, NX, A, NA,
        actions, last_atom_mask
    )


ACTION_NAMES = {0: "APPEND", 1: "CONNECT", 2: "END"}

ACTION_TEST_SMILES = [
    "CCO",
    "c1ccccc1",
    "CC(C)O",
    "C1CCCCC1",
]

for i, smiles in enumerate(ACTION_TEST_SMILES, 1):
    graph, atom_ids, ranks, bonds, bond_ids = (
        get_graph_from_smiles(smiles)
    )

    steps, _ = traverse_graph(
        graph,
        ranks,
        p=P_ALPHA,
        rng=random.Random(200 + i)
    )

    X_0, A_0 = single_reorder(
        atom_ids,
        np.column_stack([bonds, bond_ids]),
        steps
    )

    action_0, X, NX, A, NA, actions, mask = (
        single_expand(X_0, A_0)
    )

    print(f"\n{'-' * 65}")
    print(f"MOLECULE {i}: {smiles}")
    print(f"{'-' * 65}")

    print(f"Initial atom ID     : {int(action_0[0])}")
    print(f"Total states        : {len(NX)}")
    print(f"Flattened atom rows : {len(X)}")
    print(f"Flattened bond rows : {len(A)}")

    print("\nACTION SEQUENCE")
    print(
        f"{'Step':>4} {'Action':>9} {'Atoms':>6} "
        f"{'Bonds':>6} {'AtomID':>7} {'BondID':>7} "
        f"{'AppendPos':>9} {'ConnectPos':>10}"
    )

    for j, action in enumerate(actions):
        action_type, atom_type, bond_type, ap, cp = (
            map(int, action)
        )

        print(
            f"{j:4d} {ACTION_NAMES[action_type]:>9} "
            f"{NX[j]:6d} {NA[j]:6d} "
            f"{atom_type:7d} {bond_type:7d} "
            f"{ap:9d} {cp:10d}"
        )

    assert int(np.sum(actions[:, 0] == 0)) == len(X_0) - 1
    assert int(np.sum(actions[:, 0] == 1)) == (
        len(A_0) - len(X_0) + 1
    )
    assert int(np.sum(actions[:, 0] == 2)) == 1
    assert int(actions[-1, 0]) == 2

    print("\nAction sequence: PASS")

print("\nSTATUS: PASS — intermediate states and targets verified.")

9.2 — GRAPH STATE AND ACTION EXPANSION

-----------------------------------------------------------------
MOLECULE 1: CCO
-----------------------------------------------------------------
Initial atom ID     : 6
Total states        : 3
Flattened atom rows : 6
Flattened bond rows : 3

ACTION SEQUENCE
Step    Action  Atoms  Bonds  AtomID  BondID AppendPos ConnectPos
   0    APPEND      1      0       6       1         0          0
   1    APPEND      2      1      32       1         1          0
   2       END      3      2       0       0         0          0

Action sequence: PASS

-----------------------------------------------------------------
MOLECULE 2: c1ccccc1
-----------------------------------------------------------------
Initial atom ID     : 6
Total states        : 7
Flattened atom rows : 27
Flattened bond rows : 21

ACTION SEQUENCE
Step    Action  Atoms  Bonds  AtomID  BondID AppendPos ConnectPos
   0    APPEND      1      0       6       0         0          0
   1    APP

In [47]:
from collections import Counter
from itertools import islice

print("=" * 65)
print("9.3 — FULL ACTION REPLAY AND ROUND-TRIP AUDIT")
print("=" * 65)

def replay_action_path(smiles, rng):
    graph, atom_ids, ranks, bonds, bond_ids = (
        get_graph_from_smiles(smiles)
    )

    steps, log_q = traverse_graph(
        graph, ranks, p=P_ALPHA, rng=rng
    )

    X_0, A_0 = single_reorder(
        atom_ids,
        np.column_stack([bonds, bond_ids]),
        steps
    )

    action_0, X, NX, A, NA, actions, mask = (
        single_expand(X_0, A_0)
    )

    # Build the molecule only from encoded actions.
    replay = Chem.RWMol()
    replay.AddAtom(MOL_SPEC.index_to_atom(int(action_0[0])))

    x_offset = 0
    a_offset = 0

    for step, action in enumerate(actions):
        n_atoms = int(NX[step])
        n_bonds = int(NA[step])

        # Verify graph state before executing this action.
        assert replay.GetNumAtoms() == n_atoms
        assert replay.GetNumBonds() == n_bonds

        assert np.array_equal(
            X[x_offset:x_offset + n_atoms],
            X_0[:n_atoms]
        )

        assert np.array_equal(
            A[a_offset:a_offset + n_bonds],
            A_0[:n_bonds]
        )

        state_mask = mask[x_offset:x_offset + n_atoms]

        assert np.count_nonzero(state_mask) == 1

        expected_marker = (
            1 if step == 0 or actions[step - 1, 0] == 0
            else 2
        )

        assert state_mask[-1] == expected_marker

        x_offset += n_atoms
        a_offset += n_bonds

        action_type, atom_type, bond_type, ap, cp = (
            map(int, action)
        )

        if action_type == 0:
            # APPEND
            assert 0 <= ap < n_atoms
            new_atom = MOL_SPEC.index_to_atom(atom_type)
            new_index = replay.AddAtom(new_atom)

            assert new_index == n_atoms

            MOL_SPEC.index_to_bond(
                replay, ap, new_index, bond_type
            )

        elif action_type == 1:
            # CONNECT
            last_atom = n_atoms - 1

            assert 0 <= cp < last_atom
            assert replay.GetBondBetweenAtoms(
                last_atom, cp
            ) is None

            MOL_SPEC.index_to_bond(
                replay, last_atom, cp, bond_type
            )

        elif action_type == 2:
            # END
            assert step == len(actions) - 1

        else:
            raise ValueError(f"Unknown action: {action_type}")

    assert x_offset == len(X)
    assert a_offset == len(A)

    assert replay.GetNumAtoms() == len(X_0)
    assert replay.GetNumBonds() == len(A_0)

    reconstructed = replay.GetMol()
    Chem.SanitizeMol(reconstructed)

    original = Chem.MolFromSmiles(smiles)

    original_smiles = Chem.MolToSmiles(
        original, canonical=True, isomericSmiles=False
    )
    reconstructed_smiles = Chem.MolToSmiles(
        reconstructed, canonical=True, isomericSmiles=False
    )

    assert original_smiles == reconstructed_smiles, (
        f"Reconstruction mismatch:\n"
        f"Original: {original_smiles}\n"
        f"Replayed: {reconstructed_smiles}"
    )

    return {
        "atoms": len(X_0),
        "bonds": len(A_0),
        "actions": len(actions),
        "append": int(np.sum(actions[:, 0] == 0)),
        "connect": int(np.sum(actions[:, 0] == 1)),
        "end": int(np.sum(actions[:, 0] == 2)),
        "log_q": log_q,
    }


with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    replay_smiles = [
        line.strip() for line in islice(f, 50)
    ]

rng = random.Random(2026)
replay_stats = Counter()
replay_failures = []

for molecule_index, smiles in enumerate(replay_smiles, 1):
    for path_index in range(K_PATHS):
        replay_stats["attempted_paths"] += 1

        try:
            result = replay_action_path(smiles, rng)

            replay_stats["successful_paths"] += 1
            replay_stats["append_actions"] += result["append"]
            replay_stats["connect_actions"] += result["connect"]
            replay_stats["end_actions"] += result["end"]

        except Exception as exc:
            replay_stats["failed_paths"] += 1

            if len(replay_failures) < 10:
                replay_failures.append({
                    "molecule_index": molecule_index,
                    "path_index": path_index,
                    "smiles": smiles[:120],
                    "error": str(exc),
                })

print("\nREPLAY RESULTS")
print(f"Molecules tested       : {len(replay_smiles)}")
print(f"Paths per molecule     : {K_PATHS}")
print(f"Attempted paths        : {replay_stats['attempted_paths']}")
print(f"Successful paths       : {replay_stats['successful_paths']}")
print(f"Failed paths           : {replay_stats['failed_paths']}")

print("\nACTION TOTALS")
print(f"Append actions         : {replay_stats['append_actions']:,}")
print(f"Connect actions        : {replay_stats['connect_actions']:,}")
print(f"End actions            : {replay_stats['end_actions']:,}")

print("\nFAILURE EXAMPLES")
for failure in replay_failures:
    print(failure)
if not replay_failures:
    print("None")

print("\nSTATUS")
if replay_stats["failed_paths"] == 0:
    print("PASS — all sampled paths reconstructed correctly.")
else:
    print("FAIL — action reconstruction requires investigation.")

assert replay_stats["failed_paths"] == 0
assert replay_stats["successful_paths"] == 50 * K_PATHS

9.3 — FULL ACTION REPLAY AND ROUND-TRIP AUDIT

REPLAY RESULTS
Molecules tested       : 50
Paths per molecule     : 5
Attempted paths        : 250
Successful paths       : 250
Failed paths           : 0

ACTION TOTALS
Append actions         : 5,740
Connect actions        : 770
End actions            : 250

FAILURE EXAMPLES
None

STATUS
PASS — all sampled paths reconstructed correctly.


## Dataset Loader and Graph Batching

In [48]:
import random
import hashlib
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

print("=" * 65)
print("10.1 — DATASET LOADING AND CONFIGURATION")
print("=" * 65)

# Training configuration
BATCH_SIZE = 32
K_PATHS = 5
P_ALPHA = 0.8
SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Load the already-validated dataset.
with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    ALL_SMILES = [line.strip() for line in f]

# Load saved train/validation indices.
with np.load(SPLIT_NPZ_PATH, allow_pickle=False) as saved:
    TRAIN_INDICES = saved["train_indices"]
    VAL_INDICES = saved["val_indices"]

assert len(ALL_SMILES) == 1_090_506
assert len(TRAIN_INDICES) == 1_035_981
assert len(VAL_INDICES) == 54_525

class SMILESDataset(Dataset):
    def __init__(self, smiles, indices):
        self.smiles = smiles
        self.indices = np.asarray(indices, dtype=np.int32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return self.smiles[int(self.indices[index])]

train_dataset = SMILESDataset(ALL_SMILES, TRAIN_INDICES)
val_dataset = SMILESDataset(ALL_SMILES, VAL_INDICES)

print("\nDATASET")
print(f"Training molecules   : {len(train_dataset):,}")
print(f"Validation molecules : {len(val_dataset):,}")

print("\nCONFIGURATION")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Paths per molecule   : {K_PATHS}")
print(f"Paths per batch      : {BATCH_SIZE * K_PATHS}")
print(f"Alpha                : {P_ALPHA}")
print(f"Device               : {DEVICE}")
print(f"Atom types           : {MOL_SPEC.num_atom_types}")
print(f"Bond types           : {MOL_SPEC.num_bond_types}")

print("\nSTATUS: Dataset loaded successfully.")

10.1 — DATASET LOADING AND CONFIGURATION

DATASET
Training molecules   : 1,035,981
Validation molecules : 54,525

CONFIGURATION
Batch size           : 32
Paths per molecule   : 5
Paths per batch      : 160
Alpha                : 0.8
Device               : cuda
Atom types           : 55
Bond types           : 4

STATUS: Dataset loaded successfully.


In [49]:
from scipy import sparse

print("=" * 65)
print("10.2 — GRAPH COLLATOR AND GRU INDEX CONSTRUCTION")
print("=" * 65)

def build_relation_edges(global_bonds, n_nodes, n_bond_types=4):
    """
    CDGCN adjacency channels:
      0–3: aromatic, single, double, triple
      4: length-2 walks
      5: length-3 walks, excluding direct neighbors

    Matches the previous notebook's adjacency construction.
    """

    pair = global_bonds[:, :2]

    rows = np.concatenate([pair[:, 0], pair[:, 1]])
    cols = np.concatenate([pair[:, 1], pair[:, 0]])

    adj = sparse.coo_matrix(
        (np.ones(len(rows), dtype=np.int32), (rows, cols)),
        shape=(n_nodes, n_nodes)
    ).tocsr()

    adj.data[:] = 1

    d2 = adj @ adj
    d3 = d2 @ adj
    d3 = d3 - d3.multiply(adj)

    edge_channels = []

    # Four bond-type channels.
    for bond_type in range(n_bond_types):
        undirected = global_bonds[
            global_bonds[:, 2] == bond_type, :2
        ]

        directed = np.concatenate(
            [undirected, undirected[:, ::-1]], axis=0
        )

        edge_channels.append(
            torch.from_numpy(directed.T.copy()).long()
        )

    # Two distance-based channels.
    for matrix in (d2, d3):
        upper = sparse.triu(matrix, k=1).tocoo()

        edges = np.stack(
            [upper.row, upper.col], axis=1
        ).astype(np.int64)

        directed = np.concatenate(
            [edges, edges[:, ::-1]], axis=0
        )

        edge_channels.append(
            torch.from_numpy(directed.T.copy()).long()
        )

    return edge_channels


def build_graph_batch(smiles_batch, k=K_PATHS, alpha=P_ALPHA, seed=None):
    """
    Build K generation paths for each molecule.

    Outputs flattened graph states, node-to-state mappings,
    GRU sequence mappings and global action targets.
    """

    if not smiles_batch:
        raise ValueError("Cannot build an empty batch")

    all_x = []
    all_masks = []
    all_edges = []
    all_actions = []
    all_nx = []

    node_to_state = []
    state_to_path = []
    state_to_time = []

    init_ids = []
    log_qs = []
    path_lengths = []
    path_state_starts = []

    total_nodes = 0
    total_states = 0

    for molecule_id, smiles in enumerate(smiles_batch):

        graph, atom_ids, ranks, bonds, bond_ids = (
            get_graph_from_smiles(smiles)
        )

        A0 = np.column_stack((
            np.asarray(bonds, dtype=np.int32),
            np.asarray(bond_ids, dtype=np.int32)
        ))

        # Reproducible paths for validation/sanity checks.
        if seed is None:
            mol_rng = random
        else:
            stable_hash = hashlib.blake2b(
                f"{seed}:{smiles}".encode(),
                digest_size=8
            ).digest()

            mol_rng = random.Random(
                int.from_bytes(stable_hash, "little")
            )

        for path_index in range(k):

            order, log_q = traverse_graph(
                graph, ranks, p=alpha, rng=mol_rng
            )

            x0, a0 = single_reorder(
                atom_ids, A0, order
            )

            init, x, nx, a, na, actions, mask = (
                single_expand(x0, a0)
            )

            n_states = len(nx)

            assert n_states <= MAX_ITER_PER_MOL

            # Node offsets for every intermediate graph.
            state_offsets = total_nodes + np.concatenate([
                [0], np.cumsum(nx[:-1])
            ])

            state_ids = (
                total_states + np.arange(n_states)
            )

            # Convert state-local bonds to global node indices.
            a = a.copy()
            a[:, :2] += np.repeat(
                state_offsets, na
            )[:, None]

            # Convert action positions to global node indices.
            actions = actions.copy()

            is_append = actions[:, 0] == 0
            is_connect = actions[:, 0] == 1

            actions[is_append, 3] += state_offsets[is_append]
            actions[is_connect, 4] += state_offsets[is_connect]

            all_x.append(x)
            all_masks.append(mask)
            all_edges.append(a)
            all_actions.append(actions)
            all_nx.append(nx)

            node_to_state.append(
                np.repeat(state_ids, nx)
            )

            global_path_id = molecule_id * k + path_index

            state_to_path.extend(
                [global_path_id] * n_states
            )

            state_to_time.extend(
                range(n_states)
            )

            init_ids.append(int(init[0]))
            log_qs.append(log_q)
            path_lengths.append(n_states)
            path_state_starts.append(total_states)

            total_nodes += len(x)
            total_states += n_states

    X = np.concatenate(all_x).astype(np.int64)
    bonds = np.concatenate(all_edges).astype(np.int64)
    NX = np.concatenate(all_nx).astype(np.int64)
    actions = np.concatenate(all_actions).astype(np.int64)

    # Pad only to this batch's maximum sequence length.
    max_time = max(path_lengths)

    rnn_gather = np.zeros(
        (len(path_lengths), max_time),
        dtype=np.int64
    )

    for path_id, (start, length) in enumerate(
        zip(path_state_starts, path_lengths)
    ):
        rnn_gather[path_id, :length] = np.arange(
            start, start + length
        )

    assert len(X) == total_nodes
    assert len(NX) == total_states
    assert len(actions) == total_states
    assert len(state_to_path) == total_states
    assert np.all(np.isfinite(log_qs))

    return {
        "X": torch.from_numpy(X),

        "mask": torch.from_numpy(
            np.concatenate(all_masks).astype(np.int64)
        ),

        "edges": build_relation_edges(
            bonds, len(X), MOL_SPEC.num_bond_types
        ),

        "NX": torch.from_numpy(NX),

        "node_to_state": torch.from_numpy(
            np.concatenate(node_to_state).astype(np.int64)
        ),

        "state_to_path": torch.tensor(
            state_to_path, dtype=torch.long
        ),

        "state_to_time": torch.tensor(
            state_to_time, dtype=torch.long
        ),

        "rnn_gather": torch.from_numpy(rnn_gather),

        "actions": torch.from_numpy(actions),

        "init_ids": torch.tensor(
            init_ids, dtype=torch.long
        ),

        "log_q": torch.tensor(
            log_qs, dtype=torch.float32
        ).reshape(len(smiles_batch), k),

        "batch_size": len(smiles_batch),
        "k": k,
    }


def batch_to_device(batch, device):
    """Transfer tensor fields to CPU/GPU."""
    return {
        key: (
            [item.to(device) for item in value]
            if isinstance(value, list)
            else value.to(device)
            if isinstance(value, torch.Tensor)
            else value
        )
        for key, value in batch.items()
    }


def collate_train(records):
    return build_graph_batch(records, seed=None)


def collate_val(records):
    return build_graph_batch(records, seed=SEED)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    collate_fn=collate_train,
    generator=torch.Generator().manual_seed(SEED),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    collate_fn=collate_val,
)

print("\nDATALOADER CONFIGURATION")
print(f"Training batches/epoch : {len(train_loader):,}")
print(f"Validation batches     : {len(val_loader):,}")
print(f"Batch size             : {BATCH_SIZE}")
print(f"Paths per batch        : {BATCH_SIZE * K_PATHS}")
print(f"Adjacency channels     : {MOL_SPEC.num_bond_types + RECEPTIVE_FIELD}")
print("Training path sampling : Stochastic")
print("Validation paths       : Deterministic")

print("\nSTATUS: Graph collator and DataLoaders ready.")

10.2 — GRAPH COLLATOR AND GRU INDEX CONSTRUCTION

DATALOADER CONFIGURATION
Training batches/epoch : 32,374
Validation batches     : 1,704
Batch size             : 32
Paths per batch        : 160
Adjacency channels     : 6
Training path sampling : Stochastic
Validation paths       : Deterministic

STATUS: Graph collator and DataLoaders ready.


In [50]:
print("=" * 65)
print("10.3 — GRAPH BATCH SANITY CHECK")
print("=" * 65)

sample_smiles = [
    train_dataset[i] for i in range(4)
]

test_batch = build_graph_batch(
    sample_smiles,
    k=K_PATHS,
    alpha=P_ALPHA,
    seed=2026
)

assert test_batch["batch_size"] == 4
assert test_batch["k"] == 5
assert test_batch["log_q"].shape == (4, 5)

assert len(test_batch["edges"]) == 6
assert len(test_batch["actions"]) == len(test_batch["NX"])

assert len(test_batch["X"]) == len(test_batch["mask"])

assert test_batch["rnn_gather"].shape[0] == 20

assert train_loader.batch_size == 32

print("\nBATCH SHAPES")
for key in [
    "X", "mask", "NX", "node_to_state",
    "actions", "rnn_gather", "log_q"
]:
    print(f"{key:20s}: {tuple(test_batch[key].shape)}")

print("\nGRAPH RELATIONS")
for i, edges in enumerate(test_batch["edges"]):
    print(f"Relation {i}: {edges.shape[1]:,} directed edges")

print("\nSUMMARY")
print(f"Molecules                : {test_batch['batch_size']}")
print(f"Sampled paths            : {test_batch['batch_size'] * K_PATHS}")
print(f"Intermediate graph states: {len(test_batch['NX']):,}")
print(f"Flattened atoms          : {len(test_batch['X']):,}")
print(f"Training batch size      : {BATCH_SIZE}")

print("\nSTATUS: PASS — dataset batching verified.")

10.3 — GRAPH BATCH SANITY CHECK

BATCH SHAPES
X                   : (8739,)
mask                : (8739,)
NX                  : (595,)
node_to_state       : (8739,)
actions             : (595, 5)
rnn_gather          : (20, 36)
log_q               : (4, 5)

GRAPH RELATIONS
Relation 0: 8,634 directed edges
Relation 1: 7,522 directed edges
Relation 2: 1,534 directed edges
Relation 3: 0 directed edges
Relation 4: 21,824 directed edges
Relation 5: 21,744 directed edges

SUMMARY
Molecules                : 4
Sampled paths            : 20
Intermediate graph states: 595
Flattened atoms          : 8,739
Training batch size      : 32

STATUS: PASS — dataset batching verified.


## CDGCN Neural Network Architecture and Forward Pass

In [51]:
import torch.nn as nn
import torch.nn.functional as F

print("=" * 65)
print("11.1 — GRAPH CONVOLUTIONAL BACKBONE")
print("=" * 65)


def segment_sum(x, ids, n):
    result = x.new_zeros((n,) + x.shape[1:])
    return result.index_add(0, ids, x)


class LinearBN(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.linear = nn.Linear(din, dout, bias=False)
        self.bn = nn.BatchNorm1d(dout)

    def forward(self, x):
        return self.bn(self.linear(x))


class GraphConv(nn.Module):
    def __init__(self, din, dout, n_relations):
        super().__init__()

        self.W = nn.Parameter(
            torch.empty(din * (1 + n_relations), dout)
        )
        nn.init.xavier_uniform_(self.W)

        self.n_relations = n_relations

    def forward(self, x, edges):
        assert len(edges) == self.n_relations

        terms = [x]

        for edge in edges:
            aggregate = torch.zeros_like(x)

            if edge.numel() > 0:
                aggregate = aggregate.index_add(
                    0, edge[0], x[edge[1]]
                )

            terms.append(aggregate)

        return torch.cat(terms, dim=1) @ self.W


class GraphBackbone(nn.Module):
    def __init__(
        self, n_atom, n_bond, D,
        F_e, F_h, F_skip, F_c
    ):
        super().__init__()

        self.atom_embedding = nn.Embedding(n_atom, F_e)
        self.mask_embedding = nn.Embedding(3, F_e)

        self.convs = nn.ModuleList()
        self.pre_bns = nn.ModuleList()

        for i, out_dim in enumerate(F_h):
            in_dim = F_e if i == 0 else F_h[i - 1]

            self.pre_bns.append(
                nn.Identity()
                if i == 0
                else nn.BatchNorm1d(in_dim)
            )

            self.convs.append(
                GraphConv(in_dim, out_dim, n_bond + D)
            )

        self.bn_skip = nn.BatchNorm1d(sum(F_h))
        self.skip = LinearBN(sum(F_h), F_skip)

        self.dense = nn.ModuleList()

        for din, dout in zip(
            [F_skip] + F_c[:-1], F_c
        ):
            self.dense.append(LinearBN(din, dout))

    def forward(self, batch):
        x = (
            self.atom_embedding(batch["X"])
            + self.mask_embedding(batch["mask"])
        )

        layer_outputs = []

        for conv, bn in zip(self.convs, self.pre_bns):

            x = conv(
                F.relu(bn(x))
                if not isinstance(bn, nn.Identity)
                else x,
                batch["edges"]
            )

            layer_outputs.append(x)

        x = torch.cat(layer_outputs, dim=1)
        x = F.relu(self.bn_skip(x))
        x = F.relu(self.skip(x))

        for dense in self.dense:
            x = F.relu(dense(x))

        return x


print("GCN layers          : 6")
print("Atom embedding      : 16")
print("GCN widths          : [32, 64, 128, 128, 256, 256]")
print("Skip projection     : 256")
print("Final node embedding: 512")
print("Graph relations     : 6")

print("\nSTATUS: Graph backbone defined.")

11.1 — GRAPH CONVOLUTIONAL BACKBONE
GCN layers          : 6
Atom embedding      : 16
GCN widths          : [32, 64, 128, 128, 256, 256]
Skip projection     : 256
Final node embedding: 512
Graph relations     : 6

STATUS: Graph backbone defined.


In [52]:
import math

print("=" * 65)
print("11.2 — GENERATOR, POLICY AND NLL")
print("=" * 65)


class Policy(nn.Module):
    def __init__(self, fin, hidden, n_atom, n_bond):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        self.linear_h = LinearBN(2 * fin, hidden)
        self.linear_h_t = LinearBN(fin, hidden)

        self.linear_x = nn.Linear(
            hidden, n_bond * (n_atom + 1)
        )

        self.linear_x_t = nn.Linear(hidden, 1)

    def forward(self, x, batch, h_state):

        node_to_state = batch["node_to_state"]

        node_features = torch.cat(
            [x, h_state[node_to_state]],
            dim=1
        )

        node_logits = self.linear_x(
            F.relu(self.linear_h(node_features))
        )

        end_logits = self.linear_x_t(
            F.relu(self.linear_h_t(h_state))
        ).squeeze(-1)

        # Stable joint log-softmax over all node actions + END.
        state_max = end_logits.clone()

        state_max = state_max.scatter_reduce(
            0,
            node_to_state,
            node_logits.max(dim=1).values,
            reduce="amax",
            include_self=True
        )

        shifted_nodes = torch.exp(
            node_logits - state_max[node_to_state, None]
        )

        shifted_end = torch.exp(
            end_logits - state_max
        )

        normalizer = (
            segment_sum(
                shifted_nodes.sum(dim=1),
                node_to_state,
                len(end_logits)
            )
            + shifted_end
        )

        log_z = state_max + torch.log(normalizer)

        node_logp = (
            node_logits - log_z[node_to_state, None]
        )

        end_logp = end_logits - log_z

        return node_logp, end_logp


class CDGCNPretrainer(nn.Module):
    def __init__(
        self,
        n_atom,
        n_bond,
        D=2,
        F_e=16,
        F_h=(32, 64, 128, 128, 256, 256),
        F_skip=256,
        F_c=(512,),
        F_policy=128,
        n_rnn=3
    ):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        self.backbone = GraphBackbone(
            n_atom, n_bond, D,
            F_e, list(F_h), F_skip, list(F_c)
        )

        self.rnn = nn.GRU(
            input_size=2 * F_c[-1],
            hidden_size=F_c[-1],
            num_layers=n_rnn,
            batch_first=True
        )

        self.policy = Policy(
            F_c[-1], F_policy, n_atom, n_bond
        )

        # Unconditional initial-atom distribution.
        self.init_atom_logits = nn.Parameter(
            torch.zeros(n_atom)
        )

        # Parameter initialization.
        for module in self.modules():

            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)

                if getattr(module, "bias", None) is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.GRU):
                for name, param in module.named_parameters():
                    if "weight" in name:
                        nn.init.xavier_uniform_(param)
                    else:
                        nn.init.zeros_(param)

            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, batch, return_debug=False):

        B = batch["batch_size"]
        K = batch["k"]

        # 1. Graph node embeddings.
        x = self.backbone(batch)

        n_states = batch["NX"].numel()

        # 2. Mean pooling over each intermediate graph.
        pooled = segment_sum(
            x, batch["node_to_state"], n_states
        ) / batch["NX"][:, None]

        # 3. Last appended atom embedding.
        last_indices = torch.cumsum(
            batch["NX"], dim=0
        ) - 1

        last = x[last_indices]

        # 4. GRU input: graph embedding + last atom.
        features = torch.cat([pooled, last], dim=1)

        padded = features[batch["rnn_gather"]]

        rnn_output, _ = self.rnn(padded)

        h_state = rnn_output[
            batch["state_to_path"],
            batch["state_to_time"]
        ]

        # 5. Joint action log-probabilities.
        node_logp, end_logp = self.policy(
            x, batch, h_state
        )

        actions = batch["actions"]
        action_type = actions[:, 0]

        selected = end_logp.clone()

        append_rows = torch.where(action_type == 0)[0]
        connect_rows = torch.where(action_type == 1)[0]

        if len(append_rows):
            action_indices = (
                self.n_bond
                + actions[append_rows, 1] * self.n_bond
                + actions[append_rows, 2]
            )

            selected[append_rows] = node_logp[
                actions[append_rows, 3],
                action_indices
            ]

        if len(connect_rows):
            selected[connect_rows] = node_logp[
                actions[connect_rows, 4],
                actions[connect_rows, 2]
            ]

        # 6. Sum log-probabilities along each path.
        path_ll = segment_sum(
            selected,
            batch["state_to_path"],
            B * K
        )

        init_logp = F.log_softmax(
            self.init_atom_logits, dim=0
        )

        path_ll = path_ll + init_logp[batch["init_ids"]]

        # 7. Importance sampling: log P - log Q.
        corrected = (
            path_ll.reshape(B, K) - batch["log_q"]
        )

        per_molecule_ll = (
            torch.logsumexp(corrected, dim=1)
            - math.log(K)
        )

        if not return_debug:
            return per_molecule_ll

        # Diagnostic: probabilities must sum to one.
        probability_mass = (
            segment_sum(
                node_logp.exp().sum(dim=1),
                batch["node_to_state"],
                n_states
            )
            + end_logp.exp()
        )

        debug = {
            "max_probability_error": (
                probability_mass - 1
            ).abs().max().item(),

            "mean_path_logp": path_ll.mean().item(),

            "mean_log_q": batch["log_q"].mean().item(),

            "max_sequence_length": padded.shape[1]
        }

        return per_molecule_ll, debug


def cdgcn_nll(per_molecule_ll):
    """Standard unconditional negative log-likelihood."""

    assert per_molecule_ll.ndim == 1
    assert torch.isfinite(per_molecule_ll).all()

    return -per_molecule_ll.mean()


print("Architecture  : CDGCN unconditional generator")
print("Objective     : Negative importance-sampled log-likelihood")
print("Toxicity loss : None")
print("Policy        : Stable joint log-softmax")
print("RNN           : 3-layer GRU")
print("Importance K  :", K_PATHS)

print("\nSTATUS: Generator and NLL defined.")

11.2 — GENERATOR, POLICY AND NLL
Architecture  : CDGCN unconditional generator
Objective     : Negative importance-sampled log-likelihood
Toxicity loss : None
Policy        : Stable joint log-softmax
RNN           : 3-layer GRU
Importance K  : 5

STATUS: Generator and NLL defined.


In [53]:
print("=" * 65)
print("11.3 — COMPLETE MODEL SANITY CHECK")
print("=" * 65)

MODEL_CONFIG = {
    "n_atom": MOL_SPEC.num_atom_types,
    "n_bond": MOL_SPEC.num_bond_types,
    "D": RECEPTIVE_FIELD,
    "F_e": 16,
    "F_h": (32, 64, 128, 128, 256, 256),
    "F_skip": 256,
    "F_c": (512,),
    "F_policy": 128,
    "n_rnn": 3,
}

torch.manual_seed(SEED)

model = CDGCNPretrainer(
    **MODEL_CONFIG
).to(DEVICE)

parameter_count = sum(
    p.numel() for p in model.parameters()
)

print("\nMODEL INFORMATION")
print(f"Device             : {DEVICE}")
print(f"Trainable parameters: {parameter_count:,}")
print(f"Atom types         : {MODEL_CONFIG['n_atom']}")
print(f"Bond types         : {MODEL_CONFIG['n_bond']}")
print(f"Training batch size: {BATCH_SIZE}")

# Small deterministic diagnostic batch.
smoke_smiles = [
    train_dataset[i] for i in range(2)
]

smoke_batch = build_graph_batch(
    smoke_smiles,
    seed=2026
)

smoke_batch = batch_to_device(
    smoke_batch, DEVICE
)

model.train()
model.zero_grad(set_to_none=True)

per_molecule_ll, debug = model(
    smoke_batch,
    return_debug=True
)

loss = cdgcn_nll(per_molecule_ll)

assert torch.isfinite(loss)
assert per_molecule_ll.shape == (2,)

loss.backward()

gradient_count = 0
gradient_norm_sq = 0.0
nonfinite_gradients = 0

for parameter in model.parameters():
    if parameter.grad is not None:
        gradient_count += 1

        grad = parameter.grad.detach()

        if not torch.isfinite(grad).all():
            nonfinite_gradients += 1
        else:
            gradient_norm_sq += grad.float().pow(2).sum().item()

gradient_norm = math.sqrt(gradient_norm_sq)

print("\nFORWARD PASS")
print(f"Molecules tested       : {smoke_batch['batch_size']}")
print(f"Paths per molecule     : {smoke_batch['k']}")
print(f"Per-molecule log-lik.  : {per_molecule_ll.detach().cpu().tolist()}")
print(f"Initial NLL            : {loss.item():.6f}")

print("\nPROBABILITY CHECK")
print(f"Maximum normalization error: {debug['max_probability_error']:.8f}")
print(f"Mean path log-probability : {debug['mean_path_logp']:.6f}")
print(f"Mean proposal log Q       : {debug['mean_log_q']:.6f}")

print("\nBACKWARD PASS")
print(f"Parameters with gradients: {gradient_count}")
print(f"Global gradient norm     : {gradient_norm:.6f}")
print(f"Non-finite gradients     : {nonfinite_gradients}")

assert debug["max_probability_error"] < 1e-4
assert nonfinite_gradients == 0
assert gradient_count > 0
assert gradient_norm > 0

if DEVICE.type == "cuda":
    print(
        f"\nPeak allocated GPU memory: "
        f"{torch.cuda.max_memory_allocated(DEVICE) / 2**30:.3f} GiB"
    )

print("\nSTATUS: PASS — full model forward/backward successful.")

# No optimizer step has occurred.
model.zero_grad(set_to_none=True)

11.3 — COMPLETE MODEL SANITY CHECK

MODEL INFORMATION
Device             : cuda
Trainable parameters: 6,976,184
Atom types         : 55
Bond types         : 4
Training batch size: 32

FORWARD PASS
Molecules tested       : 2
Paths per molecule     : 5
Per-molecule log-lik.  : [-196.02427673339844, -286.3999938964844]
Initial NLL            : 241.212128

PROBABILITY CHECK
Maximum normalization error: 0.00000083
Mean path log-probability : -255.361969
Mean proposal log Q       : -7.645149

BACKWARD PASS
Parameters with gradients: 49
Global gradient norm     : 238.281538
Non-finite gradients     : 0

Peak allocated GPU memory: 0.366 GiB

STATUS: PASS — full model forward/backward successful.


## Full Batch-32 Training Smoke Test

In [54]:
import time
import gc
import torch

print("=" * 65)
print("12.1 — OPTIMIZER AND TRAINING CONFIGURATION")
print("=" * 65)

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP_VALUE = 3.0

LR_DECAY = 1e-3
LR_DECAY_STEP = 100

SMOKE_STEPS = 3

assert BATCH_SIZE == 32
assert K_PATHS == 5
assert DEVICE.type == "cuda", "This batch-32 test requires a GPU."

print("\nOPTIMIZER")
print("Type                    : Adam")
print(f"Initial learning rate   : {LEARNING_RATE}")
print(f"Weight decay            : {WEIGHT_DECAY}")
print(f"Gradient clipping       : value +/-{GRAD_CLIP_VALUE}")
print(f"LR decay                : {LR_DECAY} every {LR_DECAY_STEP} steps")

print("\nSMOKE TEST")
print(f"Batch size              : {BATCH_SIZE}")
print(f"Paths per molecule      : {K_PATHS}")
print(f"Total paths             : {BATCH_SIZE * K_PATHS}")
print(f"Optimizer updates       : {SMOKE_STEPS}")
print("Data                    : Same fixed batch for each update")
print("Checkpoint writes       : Disabled")
print("Full training           : Disabled")

print("\nSTATUS: Configuration ready.")

12.1 — OPTIMIZER AND TRAINING CONFIGURATION

OPTIMIZER
Type                    : Adam
Initial learning rate   : 0.001
Weight decay            : 0.0
Gradient clipping       : value +/-3.0
LR decay                : 0.001 every 100 steps

SMOKE TEST
Batch size              : 32
Paths per molecule      : 5
Total paths             : 160
Optimizer updates       : 3
Data                    : Same fixed batch for each update
Checkpoint writes       : Disabled
Full training           : Disabled

STATUS: Configuration ready.


In [55]:

print("=" * 65)
print("12.2 — FULL BATCH-32 OPTIMIZER SMOKE TEST")
print("=" * 65)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE)

# Build exactly one real batch of 32 molecules.
t0 = time.perf_counter()
raw_batch = next(iter(train_loader))
collation_seconds = time.perf_counter() - t0

full_batch = batch_to_device(raw_batch, DEVICE)

assert full_batch["batch_size"] == 32
assert full_batch["k"] == 5

print("\nBATCH")
print(f"Molecules             : {full_batch['batch_size']}")
print(f"Sampled paths         : {full_batch['batch_size'] * full_batch['k']}")
print(f"Intermediate graphs   : {len(full_batch['NX']):,}")
print(f"Flattened atoms       : {len(full_batch['X']):,}")
print(f"CPU batch construction: {collation_seconds:.2f} s")

# Separate model: the original model is not modified.
smoke_model = CDGCNPretrainer(**MODEL_CONFIG).to(DEVICE)
smoke_optimizer = torch.optim.Adam(
    smoke_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

smoke_model.train()

loss_values = []
gradient_norms = []
step_times = []

print("\nOPTIMIZER STEPS")
print("-" * 65)

for step in range(1, SMOKE_STEPS + 1):
    step_start = time.perf_counter()

    smoke_optimizer.zero_grad(set_to_none=True)

    ll = smoke_model(full_batch)
    loss = cdgcn_nll(ll)

    if not torch.isfinite(loss):
        raise FloatingPointError(
            f"Non-finite NLL at step {step}"
        )

    loss.backward()

    # Measure full gradient norm before clipping.
    grad_norm = torch.nn.utils.clip_grad_norm_(
        smoke_model.parameters(),
        max_norm=float("inf"),
        error_if_nonfinite=True
    )

    # Match the previous pretraining notebook.
    torch.nn.utils.clip_grad_value_(
        smoke_model.parameters(),
        GRAD_CLIP_VALUE
    )

    smoke_optimizer.step()

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - step_start

    loss_values.append(float(loss.item()))
    gradient_norms.append(float(grad_norm.item()))
    step_times.append(elapsed)

    print(
        f"Step {step}/{SMOKE_STEPS} | "
        f"NLL {loss.item():.4f} | "
        f"Grad norm {grad_norm.item():.2f} | "
        f"Time {elapsed:.2f}s"
    )

# Evaluate the fixed batch again after the last update.
smoke_model.train()
with torch.no_grad():
    final_nll = float(
        cdgcn_nll(smoke_model(full_batch)).item()
    )

torch.cuda.synchronize()

peak_allocated = (
    torch.cuda.max_memory_allocated(DEVICE) / 2**30
)
peak_reserved = (
    torch.cuda.max_memory_reserved(DEVICE) / 2**30
)
gpu_capacity = (
    torch.cuda.get_device_properties(DEVICE).total_memory / 2**30
)

print("\n" + "=" * 65)
print("TRAINING RESULTS")
print("=" * 65)

print(f"Initial NLL            : {loss_values[0]:.4f}")
print(f"Final NLL              : {final_nll:.4f}")
print(f"NLL change             : {final_nll - loss_values[0]:+.4f}")
print(f"Mean optimizer step    : {sum(step_times)/len(step_times):.2f} s")
print(f"Peak GPU allocated     : {peak_allocated:.3f} GiB")
print(f"Peak GPU reserved      : {peak_reserved:.3f} GiB")
print(f"Total GPU capacity     : {gpu_capacity:.3f} GiB")

print("\nASSESSMENT")
if final_nll < loss_values[0]:
    print("Learning check         : PASS — fixed-batch NLL decreased")
else:
    print("Learning check         : REVIEW — fixed-batch NLL did not decrease")

if peak_reserved / gpu_capacity > 0.85:
    print("Memory check           : WARNING — limited GPU headroom")
else:
    print("Memory check           : PASS — reasonable GPU headroom")

print("\nNo checkpoints or training files were written.")

# Free temporary training objects.
del smoke_model, smoke_optimizer, full_batch, raw_batch
gc.collect()
torch.cuda.empty_cache()


12.2 — FULL BATCH-32 OPTIMIZER SMOKE TEST

BATCH
Molecules             : 32
Sampled paths         : 160
Intermediate graphs   : 4,625
Flattened atoms       : 66,407
CPU batch construction: 0.11 s

OPTIMIZER STEPS
-----------------------------------------------------------------
Step 1/3 | NLL 226.4900 | Grad norm 123.20 | Time 0.07s
Step 2/3 | NLL 196.7039 | Grad norm 118.18 | Time 0.06s
Step 3/3 | NLL 174.3909 | Grad norm 137.95 | Time 0.07s

TRAINING RESULTS
Initial NLL            : 226.4900
Final NLL              : 151.0928
NLL change             : -75.3972
Mean optimizer step    : 0.07 s
Peak GPU allocated     : 4.315 GiB
Peak GPU reserved      : 5.725 GiB
Total GPU capacity     : 44.371 GiB

ASSESSMENT
Learning check         : PASS — fixed-batch NLL decreased
Memory check           : PASS — reasonable GPU headroom

No checkpoints or training files were written.


## Training Configuration, Checkpoints and Resume

In [56]:
import os
import json
import time
import math
import random
import hashlib
import numpy as np
import torch

from pathlib import Path

print("=" * 65)
print("13.1 — FULL PRETRAINING CONFIGURATION")
print("=" * 65)

TRAIN_CONFIG = {
    "batch_size": 32,
    "k_paths": 5,
    "alpha": 0.8,
    "seed": 42,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "grad_clip_value": 3.0,

    # Inverse LR decay: gentle and explicit.
    "lr_decay": 1e-3,
    "lr_decay_every": 100,

    "target_steps": 120_000,
    "log_every": 100,
    "validate_every": 2_000,
    "save_every": 1_000,
    "validation_molecules": 2_048,
}

assert BATCH_SIZE == TRAIN_CONFIG["batch_size"]
assert K_PATHS == TRAIN_CONFIG["k_paths"]
assert P_ALPHA == TRAIN_CONFIG["alpha"]
assert len(train_dataset) == 1_035_981
assert len(val_dataset) == 54_525
assert DEVICE.type == "cuda"

TRAIN_BATCHES_PER_EPOCH = (
    len(train_dataset) // TRAIN_CONFIG["batch_size"]
)

RUN_DIR = (
    PROJECT_ROOT
    / "REWORKED_CODE"
    / "PRETRAINING_OUTPUT"
    / "no_toxicity"
    / "training_runs"
    / "chembl_B32_K5_alpha08_seed42"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = RUN_DIR / "latest.pt"
BEST_CKPT = RUN_DIR / "best.pt"
TRAIN_LOG_PATH = RUN_DIR / "training_log.jsonl"
RUN_CONFIG_PATH = RUN_DIR / "run_config.json"

# Read the saved split manifest rather than relying
# only on variables in the current Jupyter session.
with SPLIT_MANIFEST_PATH.open("r", encoding="utf-8") as f:
    split_meta = json.load(f)

assert split_meta["dataset_sha256"] == data_hash
assert split_meta["vocabulary_sha256"] == vocab_hash
assert split_meta["total_molecules"] == len(ALL_SMILES)
assert split_meta["validation_indices_sha256"] == SPLIT_HASH

# These properties may not change across a resumed run.
LOCKED_RUN_CONFIG = {
    "objective": "unconditional_importance_sampled_nll",
    "dataset_sha256": split_meta["dataset_sha256"],
    "vocabulary_sha256": split_meta["vocabulary_sha256"],
    "split_sha256": split_meta["validation_indices_sha256"],
    "model_config": MODEL_CONFIG,
    "batch_size": TRAIN_CONFIG["batch_size"],
    "k_paths": TRAIN_CONFIG["k_paths"],
    "alpha": TRAIN_CONFIG["alpha"],
    "seed": TRAIN_CONFIG["seed"],
    "learning_rate": TRAIN_CONFIG["learning_rate"],
    "weight_decay": TRAIN_CONFIG["weight_decay"],
    "grad_clip_value": TRAIN_CONFIG["grad_clip_value"],
    "lr_decay": TRAIN_CONFIG["lr_decay"],
    "lr_decay_every": TRAIN_CONFIG["lr_decay_every"],
    "validation_molecules": TRAIN_CONFIG["validation_molecules"],
    "validation_seed": 2026,
    "sampler": "epoch_seeded_permutation_drop_last",
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        LOCKED_RUN_CONFIG,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

if RUN_CONFIG_PATH.exists():
    with RUN_CONFIG_PATH.open("r", encoding="utf-8") as f:
        previous_config = json.load(f)

    assert previous_config["signature"] == RUN_SIGNATURE, (
        "Existing run has a different configuration. "
        "Use another run directory."
    )
else:
    run_record = {
        "signature": RUN_SIGNATURE,
        "locked_config": LOCKED_RUN_CONFIG,
        "initial_target_steps": TRAIN_CONFIG["target_steps"],
    }

    temporary = RUN_CONFIG_PATH.with_suffix(".json.tmp")

    with temporary.open("w", encoding="utf-8") as f:
        json.dump(run_record, f, indent=2, sort_keys=True)

    os.replace(temporary, RUN_CONFIG_PATH)

print("\nTRAINING")
print(f"Batch size             : {TRAIN_CONFIG['batch_size']}")
print(f"Paths per molecule     : {TRAIN_CONFIG['k_paths']}")
print(f"Optimizer steps        : {TRAIN_CONFIG['target_steps']:,}")
print(f"Batches per epoch      : {TRAIN_BATCHES_PER_EPOCH:,}")
print(f"Validation subset      : {TRAIN_CONFIG['validation_molecules']:,}")

print("\nOUTPUT PATHS")
print(f"Run directory          : {RUN_DIR}")
print(f"Latest checkpoint      : {LATEST_CKPT.name}")
print(f"Best checkpoint        : {BEST_CKPT.name}")
print(f"Training log           : {TRAIN_LOG_PATH.name}")

print("\nEXISTING CHECKPOINT")
print("Found" if LATEST_CKPT.exists() else "None — new run")

print("\nSTATUS: Training configuration ready.")

13.1 — FULL PRETRAINING CONFIGURATION

TRAINING
Batch size             : 32
Paths per molecule     : 5
Optimizer steps        : 120,000
Batches per epoch      : 32,374
Validation subset      : 2,048

OUTPUT PATHS
Run directory          : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42
Latest checkpoint      : latest.pt
Best checkpoint        : best.pt
Training log           : training_log.jsonl

EXISTING CHECKPOINT
None — new run

STATUS: Training configuration ready.


In [57]:
print("=" * 65)
print("13.2 — CHECKPOINT AND RESUME UTILITIES")
print("=" * 65)


def seed_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_training_components():
    model = CDGCNPretrainer(
        **MODEL_CONFIG
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )

    # LR(step) = LR0 / (1 + decay * floor(step / interval))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: (
            1.0 / (
                1.0
                + TRAIN_CONFIG["lr_decay"]
                * (step // TRAIN_CONFIG["lr_decay_every"])
            )
        ),
    )

    return model, optimizer, scheduler


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])

    if state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_training_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    completed_steps,
    best_val_nll,
):
    checkpoint = {
        "signature": RUN_SIGNATURE,
        "completed_steps": int(completed_steps),
        "best_val_nll": float(best_val_nll),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "rng_state": capture_rng_state(),
    }

    # Atomic replacement protects the previous checkpoint
    # if writing is interrupted.
    temporary = path.with_suffix(path.suffix + ".tmp")

    torch.save(checkpoint, temporary)
    os.replace(temporary, path)


def load_training_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
):
    # Only load checkpoint files created by this notebook.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    assert checkpoint["signature"] == RUN_SIGNATURE, (
        "Checkpoint configuration does not match this run."
    )

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

    restore_rng_state(checkpoint["rng_state"])

    return (
        int(checkpoint["completed_steps"]),
        float(checkpoint["best_val_nll"]),
    )


def append_training_event(event):
    with TRAIN_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event) + "\n")


def reconcile_training_log(completed_steps):
    """
    If a process crashed after logging but before checkpointing,
    discard log records beyond the recovered checkpoint step.
    """
    if not TRAIN_LOG_PATH.exists():
        return

    with TRAIN_LOG_PATH.open("r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]

    valid_lines = [
        line for line in lines
        if int(json.loads(line)["step"]) <= completed_steps
    ]

    if len(valid_lines) != len(lines):
        temporary = TRAIN_LOG_PATH.with_suffix(".jsonl.tmp")

        with temporary.open("w", encoding="utf-8") as f:
            f.writelines(valid_lines)

        os.replace(temporary, TRAIN_LOG_PATH)

        print(
            f"Reconciled training log to step {completed_steps:,}"
        )


print("Model initialization : Ready")
print("Adam optimizer        : Ready")
print("Learning-rate decay   : Ready")
print("Atomic checkpoints    : Ready")
print("Optimizer resumption  : Supported")
print("RNG restoration       : Supported")
print("Log reconciliation    : Supported")

print("\nSTATUS: Checkpoint utilities ready.")

13.2 — CHECKPOINT AND RESUME UTILITIES
Model initialization : Ready
Adam optimizer        : Ready
Learning-rate decay   : Ready
Atomic checkpoints    : Ready
Optimizer resumption  : Supported
RNG restoration       : Supported
Log reconciliation    : Supported

STATUS: Checkpoint utilities ready.


## Training Loop and Validation

In [58]:
print("=" * 65)
print("14.1 — VALIDATION AND DETERMINISTIC TRAIN SAMPLER")
print("=" * 65)

VALIDATION_SEED = 2026


@torch.no_grad()
def evaluate_pretraining(
    model,
    n_molecules=None,
):
    """
    Evaluate importance-sampled NLL on a fixed
    subset of the held-out validation dataset.

    Set n_molecules=None to evaluate all validation
    molecules.
    """

    if n_molecules is None:
        n_molecules = len(val_dataset)

    assert 1 <= n_molecules <= len(val_dataset)

    was_training = model.training
    model.eval()

    total_nll = 0.0
    total_count = 0

    try:
        for start in range(
            0, n_molecules, BATCH_SIZE
        ):
            end = min(
                start + BATCH_SIZE,
                n_molecules
            )

            smiles = [
                val_dataset[i]
                for i in range(start, end)
            ]

            batch = build_graph_batch(
                smiles,
                k=K_PATHS,
                alpha=P_ALPHA,
                seed=VALIDATION_SEED,
            )

            batch = batch_to_device(
                batch, DEVICE
            )

            ll = model(batch)

            if not torch.isfinite(ll).all():
                raise FloatingPointError(
                    "Non-finite validation likelihood."
                )

            total_nll += float(
                (-ll.double().sum()).item()
            )
            total_count += len(smiles)

    finally:
        model.train(was_training)

    return total_nll / total_count


def get_training_batch(step, cache):
    """
    Each epoch uses a reproducible permutation.

    `step` is zero-based and uniquely determines
    the selected molecule indices.

    `cache` stores the current epoch's permutation.
    """

    epoch, batch_index = divmod(
        step, TRAIN_BATCHES_PER_EPOCH
    )

    if cache.get("epoch") != epoch:

        rng = np.random.default_rng(
            TRAIN_CONFIG["seed"] + epoch
        )

        cache["permutation"] = rng.permutation(
            len(train_dataset)
        )

        cache["epoch"] = epoch

    start = batch_index * BATCH_SIZE
    end = start + BATCH_SIZE

    local_indices = cache["permutation"][
        start:end
    ]

    assert len(local_indices) == BATCH_SIZE

    return [
        train_dataset[int(i)]
        for i in local_indices
    ]


print("Validation paths       : Fixed per SMILES")
print("Validation set         : Held-out ChEMBL")
print("Training shuffle       : Deterministic per epoch")
print("Resume position        : Optimizer step")
print("Full validation        : Supported")

print("\nSTATUS: Validation and sampler ready.")

14.1 — VALIDATION AND DETERMINISTIC TRAIN SAMPLER
Validation paths       : Fixed per SMILES
Validation set         : Held-out ChEMBL
Training shuffle       : Deterministic per epoch
Resume position        : Optimizer step
Full validation        : Supported

STATUS: Validation and sampler ready.


In [59]:
import gc

print("=" * 65)
print("14.2 — PRETRAINING ENGINE")
print("=" * 65)


def run_pretraining(
    *,
    smoke=False,
    max_new_steps=None,
):
    """
    smoke=True:
        Fresh temporary model.
        No checkpoint or training-log writes.

    smoke=False:
        Resume from latest checkpoint if available.
        Otherwise initialize a fresh model.

    max_new_steps:
        Number of additional optimizer steps.
        None runs until target_steps.
    """

    seed_training(TRAIN_CONFIG["seed"])

    training_model, optimizer, scheduler = (
        create_training_components()
    )

    start_step = 0
    best_val_nll = float("inf")

    if not smoke and LATEST_CKPT.exists():

        start_step, best_val_nll = (
            load_training_checkpoint(
                LATEST_CKPT,
                training_model,
                optimizer,
                scheduler,
            )
        )

        reconcile_training_log(start_step)

        print(
            f"RESUMED from step {start_step:,}"
        )

    elif not smoke:
        reconcile_training_log(0)
        print("Starting a fresh full training run.")

    else:
        print("Starting temporary multi-batch smoke test.")

    if smoke:
        if max_new_steps is None:
            max_new_steps = 20

        assert max_new_steps > 0

        target = max_new_steps
        stop_step = max_new_steps

    else:
        target = TRAIN_CONFIG["target_steps"]

        stop_step = (
            target
            if max_new_steps is None
            else min(
                target,
                start_step + max_new_steps
            )
        )

    if start_step >= target:
        print(
            f"Training already reached target step {target:,}."
        )
        return {
            "completed_steps": start_step,
            "best_val_nll": best_val_nll,
        }

    if stop_step <= start_step:
        raise ValueError("No new training steps requested.")

    print("\nRUN CONFIGURATION")
    print(f"Start step       : {start_step:,}")
    print(f"Stop step        : {stop_step:,}")
    print(f"Target steps     : {target:,}")
    print(f"Batch size       : {BATCH_SIZE}")
    print(f"Paths per batch  : {BATCH_SIZE * K_PATHS}")

    if smoke:
        initial_val = evaluate_pretraining(
            training_model,
            n_molecules=128
        )

        print(
            f"\nInitial smoke validation NLL: "
            f"{initial_val:.4f}"
        )

    permutation_cache = {}
    training_model.train()

    last_completed = start_step
    last_loss = None
    last_grad_norm = None
    run_start_time = time.perf_counter()

    for step in range(start_step, stop_step):

        # ----------------------------------
        # 1. Load and sample generation paths
        # ----------------------------------
        smiles = get_training_batch(
            step, permutation_cache
        )

        batch = build_graph_batch(
            smiles,
            k=K_PATHS,
            alpha=P_ALPHA,
            seed=None,
        )

        batch = batch_to_device(
            batch, DEVICE
        )

        # ----------------------------------
        # 2. Forward and backward
        # ----------------------------------
        optimizer.zero_grad(set_to_none=True)

        per_molecule_ll = training_model(batch)
        loss = cdgcn_nll(per_molecule_ll)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite NLL at step {step + 1}"
            )

        loss.backward()

        # Measure the norm before value clipping.
        grad_norm = torch.nn.utils.clip_grad_norm_(
            training_model.parameters(),
            max_norm=float("inf"),
            error_if_nonfinite=True,
        )

        # Match the previous notebook's clipping.
        torch.nn.utils.clip_grad_value_(
            training_model.parameters(),
            TRAIN_CONFIG["grad_clip_value"],
        )

        # ----------------------------------
        # 3. Update model and learning rate
        # ----------------------------------
        optimizer.step()
        scheduler.step()

        completed = step + 1
        last_completed = completed

        last_loss = float(loss.detach().item())
        last_grad_norm = float(
            grad_norm.detach().item()
        )

        current_lr = optimizer.param_groups[0]["lr"]

        # ----------------------------------
        # 4. Training logs
        # ----------------------------------
        should_log = (
            smoke
            or completed == 1
            or completed % TRAIN_CONFIG["log_every"] == 0
            or completed == stop_step
        )

        if should_log:

            elapsed = (
                time.perf_counter() - run_start_time
            )

            message = (
                f"Step {completed:>7,}/{target:,} | "
                f"NLL {last_loss:9.4f} | "
                f"Grad {last_grad_norm:9.2f} | "
                f"LR {current_lr:.7f}"
            )

            print(message)

            if not smoke:
                append_training_event({
                    "event": "train",
                    "step": completed,
                    "train_nll": last_loss,
                    "grad_norm": last_grad_norm,
                    "learning_rate": current_lr,
                    "elapsed_seconds": elapsed,
                })

        # ----------------------------------
        # 5. Held-out validation
        # ----------------------------------
        should_validate = (
            completed % TRAIN_CONFIG["validate_every"] == 0
            or completed == stop_step
        )

        if should_validate:

            val_count = (
                128 if smoke
                else TRAIN_CONFIG["validation_molecules"]
            )

            val_start = time.perf_counter()

            val_nll = evaluate_pretraining(
                training_model,
                n_molecules=val_count
            )

            val_seconds = (
                time.perf_counter() - val_start
            )

            print(
                f"  VALIDATION | "
                f"NLL {val_nll:.4f} | "
                f"molecules {val_count:,} | "
                f"time {val_seconds:.1f}s"
            )

            if not smoke:

                append_training_event({
                    "event": "validation",
                    "step": completed,
                    "val_nll": val_nll,
                    "molecules": val_count,
                    "elapsed_seconds": val_seconds,
                })

                if val_nll < best_val_nll:

                    best_val_nll = val_nll

                    save_training_checkpoint(
                        BEST_CKPT,
                        training_model,
                        optimizer,
                        scheduler,
                        completed,
                        best_val_nll,
                    )

                    print(
                        "  NEW BEST validation checkpoint"
                    )

            training_model.train()

        # ----------------------------------
        # 6. Latest checkpoint
        # ----------------------------------
        should_save = (
            completed % TRAIN_CONFIG["save_every"] == 0
            or completed == stop_step
        )

        if should_save and not smoke:

            save_training_checkpoint(
                LATEST_CKPT,
                training_model,
                optimizer,
                scheduler,
                completed,
                best_val_nll,
            )

            print(
                f"  CHECKPOINT saved at step {completed:,}"
            )

    print("\n" + "=" * 65)
    print("TRAINING RUN SUMMARY")
    print("=" * 65)

    print(f"Completed step     : {last_completed:,}")
    print(f"New optimizer steps: {last_completed - start_step:,}")
    print(f"Last training NLL  : {last_loss:.4f}")
    print(f"Last gradient norm : {last_grad_norm:.4f}")

    if not smoke:
        print(f"Best validation NLL: {best_val_nll:.4f}")
        print(f"Checkpoint         : {LATEST_CKPT}")
    else:
        print("Temporary smoke model — no checkpoint saved.")

    result = {
        "completed_steps": last_completed,
        "last_training_nll": last_loss,
        "last_grad_norm": last_grad_norm,
        "best_val_nll": best_val_nll,
    }

    # The model is intentionally not returned;
    # checkpoints are the source of truth.
    del training_model, optimizer, scheduler
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


print("Forward/backward     : Implemented")
print("Optimizer and LR     : Implemented")
print("Validation           : Implemented")
print("Checkpointing        : Implemented")
print("Resume               : Implemented")
print("Training logs        : Implemented")

print("\nSTATUS: Pretraining engine ready.")

14.2 — PRETRAINING ENGINE
Forward/backward     : Implemented
Optimizer and LR     : Implemented
Validation           : Implemented
Checkpointing        : Implemented
Resume               : Implemented
Training logs        : Implemented

STATUS: Pretraining engine ready.


In [60]:
print("=" * 65)
print("14.3 — MULTI-BATCH TRAINING SMOKE TEST")
print("=" * 65)

smoke_result = run_pretraining(
    smoke=True,
    max_new_steps=20,
)

assert smoke_result["completed_steps"] == 20
assert math.isfinite(
    smoke_result["last_training_nll"]
)

print("\nSTATUS: PASS — multi-batch training completed.")

14.3 — MULTI-BATCH TRAINING SMOKE TEST
Starting temporary multi-batch smoke test.

RUN CONFIGURATION
Start step       : 0
Stop step        : 20
Target steps     : 20
Batch size       : 32
Paths per batch  : 160

Initial smoke validation NLL: 231.6657
Step       1/20 | NLL  197.6954 | Grad    105.18 | LR 0.0010000
Step       2/20 | NLL  185.0992 | Grad     92.98 | LR 0.0010000
Step       3/20 | NLL  175.9586 | Grad    109.30 | LR 0.0010000
Step       4/20 | NLL  156.4038 | Grad    109.93 | LR 0.0010000
Step       5/20 | NLL  129.5659 | Grad     89.32 | LR 0.0010000
Step       6/20 | NLL  121.0470 | Grad     66.80 | LR 0.0010000
Step       7/20 | NLL  122.4800 | Grad     69.44 | LR 0.0010000
Step       8/20 | NLL  111.7816 | Grad    105.58 | LR 0.0010000
Step       9/20 | NLL  105.5504 | Grad     58.00 | LR 0.0010000
Step      10/20 | NLL   89.4499 | Grad     54.64 | LR 0.0010000
Step      11/20 | NLL   87.0430 | Grad     71.20 | LR 0.0010000
Step      12/20 | NLL   88.2873 | Grad     73

In [61]:
print("=" * 65)
print("14.4 — FULL PRETRAINING: START / RESUME")
print("=" * 65)

# 2,000 updates for the first production run.
# Once the results look good, change this to None
# to continue to the full 120,000-step target.
STEPS_THIS_RUN = 2_000

train_result = run_pretraining(
    smoke=False,
    max_new_steps=STEPS_THIS_RUN,
)

print("\nLATEST CHECKPOINT")
print(LATEST_CKPT)

print("\nBEST CHECKPOINT")
print(BEST_CKPT)

print("\nTRAINING LOG")
print(TRAIN_LOG_PATH)

14.4 — FULL PRETRAINING: START / RESUME
Starting a fresh full training run.

RUN CONFIGURATION
Start step       : 0
Stop step        : 2,000
Target steps     : 120,000
Batch size       : 32
Paths per batch  : 160
Step       1/120,000 | NLL  197.6953 | Grad    105.18 | LR 0.0010000
Step     100/120,000 | NLL   45.7219 | Grad     40.57 | LR 0.0009990
Step     200/120,000 | NLL   36.5831 | Grad     26.98 | LR 0.0009980
Step     300/120,000 | NLL   35.5250 | Grad     26.97 | LR 0.0009970
Step     400/120,000 | NLL   32.7790 | Grad     25.75 | LR 0.0009960
Step     500/120,000 | NLL   35.1171 | Grad     38.61 | LR 0.0009950
Step     600/120,000 | NLL   31.0804 | Grad     34.27 | LR 0.0009940
Step     700/120,000 | NLL   33.2989 | Grad     23.43 | LR 0.0009930
Step     800/120,000 | NLL   32.2856 | Grad     22.03 | LR 0.0009921
Step     900/120,000 | NLL   35.0506 | Grad     24.46 | LR 0.0009911
Step   1,000/120,000 | NLL   32.1427 | Grad     25.88 | LR 0.0009901
  CHECKPOINT saved at step 1

In [64]:
print("=" * 70)
print("EXTENDED CDGCN PRETRAINING — 5 EPOCHS")
print("=" * 70)

# Five full epochs of training batches.
TOTAL_EPOCHS = 5

STEPS_PER_EPOCH = len(train_dataset) // BATCH_SIZE

TRAIN_CONFIG["target_steps"] = (
    TOTAL_EPOCHS * STEPS_PER_EPOCH
)

print(f"Training molecules : {len(train_dataset):,}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Steps per epoch    : {STEPS_PER_EPOCH:,}")
print(f"Target epochs      : {TOTAL_EPOCHS}")
print(f"Total steps        : {TRAIN_CONFIG['target_steps']:,}")

# Automatically resumes from latest.pt.
final_result = run_pretraining(
    smoke=False,
    max_new_steps=None
)

print("\n" + "=" * 70)
print("FINAL TRAINING SUMMARY")
print("=" * 70)

print(f"Completed steps     : {final_result['completed_steps']:,}")
print(f"Total epochs        : {TOTAL_EPOCHS}")
print(f"Best validation NLL : {final_result['best_val_nll']:.4f}")
print(f"Best checkpoint     : {BEST_CKPT}")

assert final_result["completed_steps"] == 161_870

print("\nSUCCESS — 5 EPOCHS COMPLETED!")

EXTENDED CDGCN PRETRAINING — 5 EPOCHS
Training molecules : 1,035,981
Batch size         : 32
Steps per epoch    : 32,374
Target epochs      : 5
Total steps        : 161,870
Reconciled training log to step 6,000
RESUMED from step 6,000

RUN CONFIGURATION
Start step       : 6,000
Stop step        : 161,870
Target steps     : 161,870
Batch size       : 32
Paths per batch  : 160
Step   6,100/161,870 | NLL   28.6759 | Grad     14.21 | LR 0.0009425
Step   6,200/161,870 | NLL   26.0553 | Grad     15.09 | LR 0.0009416
Step   6,300/161,870 | NLL   25.6884 | Grad     13.46 | LR 0.0009407
Step   6,400/161,870 | NLL   27.6280 | Grad     13.76 | LR 0.0009398
Step   6,500/161,870 | NLL   27.9356 | Grad     17.53 | LR 0.0009390
Step   6,600/161,870 | NLL   29.2916 | Grad     16.06 | LR 0.0009381
Step   6,700/161,870 | NLL   30.6481 | Grad     14.72 | LR 0.0009372
Step   6,800/161,870 | NLL   27.0013 | Grad     14.45 | LR 0.0009363
Step   6,900/161,870 | NLL   28.3858 | Grad     14.18 | LR 0.0009355
S